# Milestone 1 (v2): baseline vs repair vs repair + test agent

Three systems on the same questions, the same model and the same first draft:

| System | What it does |
|---|---|
| `baseline` | one generation, no tests, no repair |
| `repair` | generation → run the **visible** tests → (supervisor → repair) × up to `MAX_RETRIES` |
| `repair_tests` | as `repair`, plus a **test agent** that writes extra tests which join the visible set |

**Every system is graded on the hidden suite** (MBPP+ / HumanEval+ extended tests, LiveCodeBench private + public
tests). With `FEEDBACK_TESTS = "public"` the repair loop only ever sees the public tests (MBPP's 3 original asserts,
HumanEval's docstring examples, LiveCodeBench's public cases). `FEEDBACK_TESTS = "hidden"` reproduces the first M1
run, where the loop was repaired against the grading suite.

**Changes from the first M1 run**
- Full MBPP+ (378) and HumanEval+ (164); LiveCodeBench added (≥ 120 problems). Runs checkpoint after every question
  and resume in the next Kaggle session.
- The test agent is a third system, so its effect is measured as a paired difference against `repair`.
- Public/hidden test split; test-agent diagnostics (validity of generated tests against the reference, caught bugs,
  false alarms).
- A fifth cost unit: model calls + test executions.
- Cost analysis rebuilt: a per-call cost model (prompt vs completion tokens), per-benchmark Spearman correlations with
  bootstrap CIs, cached replays excluded, mechanical pairs (GPU-s ↔ wall time ↔ joules) reported as power, not as
  correlations. A warm-up call keeps model-load energy out of the first measured call.
- Charts: Wilson 95% CIs on pass@1, Holm-corrected paired tests, redundant figures merged.


## 1 · Environment

In [ ]:
import platform, subprocess, shutil, os, sys

print("Python      :", sys.version.split()[0])
print("Platform    :", platform.platform())
print("CPU count   :", os.cpu_count())

# RAM
try:
    with open("/proc/meminfo") as f:
        mem = f.readline().split()
    print("System RAM  :", round(int(mem[1]) / 1024 / 1024, 1), "GB")
except Exception:
    print("System RAM  : (unavailable)")

# Platform detection (changes how files are saved and downloaded later)
IS_KAGGLE = os.path.exists("/kaggle/working")
IS_COLAB  = os.path.exists("/content") and not IS_KAGGLE
PLATFORM  = "Kaggle" if IS_KAGGLE else ("Colab" if IS_COLAB else "local/other")
print("Platform    :", PLATFORM)

# GPU
if shutil.which("nvidia-smi"):
    try:
        out = subprocess.run(
            ["nvidia-smi",
             "--query-gpu=name,memory.total,memory.free,driver_version",
             "--format=csv,noheader"],
            capture_output=True, text=True, timeout=30)
        print("GPU         :", out.stdout.strip() or "(query failed)")
    except Exception as e:
        print("GPU         : nvidia-smi error:", e)
else:
    print("GPU         : NONE DETECTED (CPU-only -> the 30B model will be very slow)")

if IS_KAGGLE:
    print("""
KAGGLE CHECKLIST
  1. Accelerator must be 'GPU T4 x2'  (2 x 16 GB = 32 GB, enough for the 30B).
     Settings -> Accelerator -> GPU T4 x2
  2. Internet must be ON, or Ollama and the datasets cannot download.
     Settings -> Internet -> On
  3. Session limit is ~9 hours and files in /kaggle/working are LOST when the
     session ends unless you press 'Save Version'. The LAST cell packages
     everything into one zip - download it before you finish.""")

## 2 · Dependencies

In [ ]:
# Install once. Safe to re-run.
import sys, subprocess

def pip_install(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

pip_install([
    "datasets>=2.16.0",
    "statsmodels>=0.14.0",
    "pandas>=2.0.0",
    "matplotlib>=3.7.0",
    "requests>=2.31.0",
    "tqdm>=4.66.0",
    "nvidia-ml-py",        # NVML: GPU power and energy
    "huggingface_hub>=0.20",  # LiveCodeBench files
    "scipy>=1.10",           # rank correlations
])
print("Dependencies installed.")

## 3 · Configuration — the only cell you should need to edit

In [ ]:
import random, os, time

T_SESSION_START = time.time()        # the session budget below is measured from here

# ---- Model -----------------------------------------------------------------
MODEL_NAME       = "qwen3-coder:30b"          # ~18 GB at Q4: needs GPU T4 x2
ALT_MODELS       = ["qwen2.5-coder:7b", "qwen2.5-coder:3b"]   # single-GPU fallbacks
SUPERVISOR_MODEL = None     # None = the code model reviews its own work (self-critique)

if SUPERVISOR_MODEL is not None and not isinstance(SUPERVISOR_MODEL, str):
    raise ValueError("SUPERVISOR_MODEL must be a model name string or None.")

# ---- Benchmarks ------------------------------------------------------------
M1_BENCHMARKS = [("mbpp", "plus"), ("humaneval", "plus"), ("livecodebench", "lite")]
# questions per benchmark; None = every question available
M1_N_PROBLEMS = {"mbpp_plus": None,        # 378
                 "humaneval_plus": None,   # 164
                 "livecodebench": None}    # every problem in the date window below
M1_MIN_PROBLEMS = 120        # floor per benchmark (n >= 45 is the McNemar floor; 120 gives power)
ALLOW_SMALL_N   = False      # True only for a quick smoke test below the floor
SEED            = 42
MBPP_PLUS_USE_EXTENDED_TESTS = True   # grade MBPP+ with the EvalPlus suite, not 3 asserts

# LiveCodeBench (code_generation_lite). Problems released after the model's training
# cutoff are the contamination-safe ones; qwen3-coder was released in mid-2025.
LCB_REPO      = "livecodebench/code_generation_lite"
LCB_FILES     = None           # None = every test*.jsonl in the repo; or e.g. ["test5.jsonl", "test6.jsonl"]
LCB_MIN_DATE  = "2025-01-01"   # keep problems with contest_date >= this
LCB_TIMEOUT_S = 60             # whole test suite per program (private suites are large)

# ---- Systems ---------------------------------------------------------------
SYSTEMS = ["baseline", "repair", "repair_tests"]
SYSTEM_LABEL = {"baseline": "baseline", "repair": "repair",
                "repair_tests": "repair + test agent"}

# What the repair loop is allowed to see. Grading ALWAYS uses the hidden suite.
#   "public" : public tests only (MBPP's 3 asserts, HumanEval docstring examples, LCB public cases)
#   "hidden" : the grading suite itself - reproduces the first M1 run (optimistic upper bound)
FEEDBACK_TESTS = "public"

# ---- Repair loop -----------------------------------------------------------
TEMPERATURE               = 0.1
MAX_RETRIES               = 3       # max supervisor -> repair rounds per question
USE_SIGNATURE_HINT        = True    # show an example call so arity/order is not guessed
FEEDBACK_MODE             = "structured"   # "structured" = failing test + got/expected
EARLY_STOP_NO_PROGRESS    = True    # stop when a repair adds no passing visible tests
COMBINE_SUPERVISOR_REPAIR = False   # True = diagnosis folded into the repair call
TEST_AGENT_MAX_TESTS      = 5       # asserts kept from the test agent per question

# ---- Generation / execution ------------------------------------------------
NUM_PREDICT     = 512       # MBPP+ / HumanEval+
NUM_PREDICT_LCB = 2048      # LiveCodeBench programs are longer
NUM_CTX         = 8192      # one value for every benchmark: changing it forces a model reload
KEEP_ALIVE      = "24h"     # never unload the 30B mid-run: a reload lands in the GPU budget
TIMEOUT_SECONDS = 15        # MBPP+ / HumanEval+ suites
M1_TIMEOUT_S    = TIMEOUT_SECONDS

# ---- Checkpointing ---------------------------------------------------------
RESUME               = True   # skip questions already in outputs_milestone1/<bench>/rows.jsonl
SESSION_BUDGET_HOURS = 8.0    # stop starting new questions after this (Kaggle kills at ~9-12 h)

# ---- Output ----------------------------------------------------------------
ON_KAGGLE     = os.path.exists("/kaggle/working")
BASE_DIR      = "/kaggle/working" if ON_KAGGLE else "."
M1_OUTPUT_DIR = os.path.join(BASE_DIR, "outputs_milestone1")
OUTPUT_DIR    = M1_OUTPUT_DIR          # the Ollama log lands here too
os.makedirs(M1_OUTPUT_DIR, exist_ok=True)

random.seed(SEED)

EXPERIMENT_CONFIG = {
    "model_name": MODEL_NAME, "supervisor_model": SUPERVISOR_MODEL,
    "benchmarks": M1_BENCHMARKS, "n_problems": M1_N_PROBLEMS, "seed": SEED,
    "systems": SYSTEMS, "feedback_tests": FEEDBACK_TESTS,
    "temperature": TEMPERATURE, "max_retries": MAX_RETRIES,
    "use_signature_hint": USE_SIGNATURE_HINT, "feedback_mode": FEEDBACK_MODE,
    "early_stop_no_progress": EARLY_STOP_NO_PROGRESS,
    "combine_supervisor_repair": COMBINE_SUPERVISOR_REPAIR,
    "test_agent_max_tests": TEST_AGENT_MAX_TESTS,
    "num_predict": NUM_PREDICT, "num_predict_lcb": NUM_PREDICT_LCB, "num_ctx": NUM_CTX,
    "timeout_seconds": TIMEOUT_SECONDS, "lcb_timeout_s": LCB_TIMEOUT_S,
    "lcb_min_date": LCB_MIN_DATE, "mbpp_plus_extended_tests": MBPP_PLUS_USE_EXTENDED_TESTS,
}
print("Results will be written to:", M1_OUTPUT_DIR)
for k, v in EXPERIMENT_CONFIG.items():
    print("  %-26s %s" % (k, v))


## 4 · Kaggle preflight
Checks internet, both GPUs, and keeps the model weights out of the 20 GB output quota.

In [ ]:
import os, shutil, subprocess

if ON_KAGGLE:
    print("Platform: KAGGLE\n")

    # --- internet (required for the model pull and the dataset) -------------
    net_ok = False
    try:
        import requests as _rq
        net_ok = _rq.get("https://huggingface.co", timeout=8).status_code < 500
    except Exception:
        net_ok = False
    print("  Internet enabled : %s" % ("YES" if net_ok else "NO"))
    if not net_ok:
        print("    -> Turn Internet ON in the right-hand Settings panel, then re-run.")

    # --- GPUs (30B needs two T4s) ------------------------------------------
    n_gpu, names = 0, ""
    if shutil.which("nvidia-smi"):
        out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                              "--format=csv,noheader"],
                             capture_output=True, text=True)
        names = out.stdout.strip()
        n_gpu = len([l for l in names.splitlines() if l.strip()])
    print("  GPUs detected    : %d" % n_gpu)
    for l in names.splitlines():
        print("     %s" % l.strip())
    if n_gpu < 2 and "30b" in MODEL_NAME.lower():
        print("    -> WARNING: %s needs ~18GB. With one GPU it will not fit."
              % MODEL_NAME)
        print("       Either set Accelerator = 'GPU T4 x2', or change MODEL_NAME")
        print("       to 'qwen2.5-coder:7b' in the configuration cell.")

    # --- keep the 18GB model OUT of /kaggle/working ------------------------
    # Anything under /kaggle/working counts toward the 20GB output limit and is
    # copied on 'Save Version'. Model weights belong in scratch space.
    scratch = "/kaggle/temp/ollama" if os.path.exists("/kaggle/temp") else "/root/.ollama"
    os.makedirs(scratch, exist_ok=True)
    os.environ["OLLAMA_MODELS"] = scratch
    print("  Model store      : %s  (scratch - not saved to Output)" % scratch)

    # --- disk headroom -----------------------------------------------------
    for path in ["/kaggle/working", scratch]:
        try:
            st = os.statvfs(path)
            print("  Free on %-16s %.1f GB" % (path + ":", st.f_bavail * st.f_frsize / 1e9))
        except Exception:
            pass
    print("\n  Session limit is ~9 hours. Check the pilot's runtime estimate")
    print("  (Section 19) before enabling the main experiment.")
    print("  Outputs will be written under /kaggle/working so they are downloadable.")
else:
    print("Not on Kaggle - preflight skipped.")

## 5 · Ollama server
Installs and starts Ollama. `OLLAMA_NUM_PARALLEL=1` is set on purpose: batched requests would
share one GPU window and make per-call energy unattributable.

In [ ]:
import subprocess, shutil, time, os, requests

OLLAMA_HOST = "http://127.0.0.1:11434"

def ollama_up():
    try:
        return requests.get(OLLAMA_HOST + "/api/tags", timeout=3).status_code == 200
    except Exception:
        return False

def _sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

if not shutil.which("ollama"):
    # 1) Ollama's installer requires zstd for extraction -> install it first.
    if not shutil.which("zstd"):
        print("Installing zstd ...")
        r = _sh("apt-get update -qq && apt-get install -y -qq zstd")
        if r.returncode != 0:                       # retry with sudo if not root
            r = _sh("sudo apt-get update -qq && sudo apt-get install -y -qq zstd")
        if not shutil.which("zstd"):
            raise RuntimeError("Could not install zstd:\n" + (r.stderr or "")[-1500:])

    # 2) now run the Ollama installer
    print("Installing Ollama ...")
    r = _sh("curl -fsSL https://ollama.com/install.sh | sh")
    print((r.stdout or "")[-600:]); print((r.stderr or "")[-600:])
else:
    print("Ollama already installed:", shutil.which("ollama"))

# resolve the binary (installer puts it in /usr/local/bin)
OLLAMA_BIN = shutil.which("ollama") or "/usr/local/bin/ollama"
if not os.path.exists(OLLAMA_BIN):
    raise RuntimeError("Ollama still not installed. Check the installer output above "
                       "and that this notebook has Internet enabled.")
print("Ollama binary:", OLLAMA_BIN)

if not ollama_up():
    print("Starting Ollama server ...")
    os.environ.setdefault("OLLAMA_HOST", "127.0.0.1:11434")
    os.environ["OLLAMA_NUM_PARALLEL"] = "1"       # one call at a time -> attributable energy
    os.environ["OLLAMA_MAX_LOADED_MODELS"] = "1"  # nothing else resident in VRAM
    os.environ["OLLAMA_KEEP_ALIVE"] = KEEP_ALIVE
    _log = open(os.path.join(OUTPUT_DIR, "ollama_serve.log"), "w")
    subprocess.Popen([OLLAMA_BIN, "serve"], stdout=_log, stderr=_log)
    for _ in range(60):
        if ollama_up():
            break
        time.sleep(1)

print("Ollama server reachable:", ollama_up())
if not ollama_up():
    raise RuntimeError("Ollama server did not come up. Check outputs/ollama_serve.log "
                       "and that this environment allows background processes.")

## 6 · Model download and verification

In [ ]:
import subprocess, requests, json, time

def list_local_models():
    try:
        r = requests.get(OLLAMA_HOST + "/api/tags", timeout=10)
        return [m["name"] for m in r.json().get("models", [])]
    except Exception:
        return []

def ensure_model(name):
    if not isinstance(name, str) or not name.strip():
        raise ValueError("ensure_model() needs a model name string, got %r" % (name,))
    local = list_local_models()
    if name in local or (name + ":latest") in local:
        print("Model present:", name)
        return True
    print("Pulling model '%s' (this can take a while / large download) ..." % name)
    p = subprocess.run(["ollama", "pull", name], capture_output=True, text=True)
    if p.returncode != 0:
        print("Pull FAILED:\n", p.stderr[-2000:])
        return False
    return name in list_local_models()

ok = ensure_model(MODEL_NAME)
if not ok:
    print("\n[!] Could not obtain '%s'." % MODEL_NAME)
    print("    Available locally:", list_local_models())
    print("    Consider setting MODEL_NAME to one of:", ALT_MODELS)
    raise RuntimeError("Model '%s' unavailable. Edit MODEL_NAME in the config cell." % MODEL_NAME)

# Confirm the model actually generates
def _quick_probe(name):
    t0 = time.time()
    r = requests.post(OLLAMA_HOST + "/api/chat", timeout=300, json={
        "model": name, "stream": False,
        "messages": [{"role": "user", "content": "Reply with the single word: OK"}],
        "options": {"temperature": 0.0, "num_predict": 8},
    })
    return r.json()["message"]["content"].strip(), time.time() - t0

resp, dt = _quick_probe(MODEL_NAME)
print("Model responded in %.1fs -> %r" % (dt, resp[:60]))

# Optional independent reviewer model
if SUPERVISOR_MODEL and SUPERVISOR_MODEL != MODEL_NAME:
    print("\nSupervisor uses a SEPARATE model:", SUPERVISOR_MODEL)
    if not ensure_model(SUPERVISOR_MODEL):
        raise RuntimeError("Supervisor model %r unavailable." % SUPERVISOR_MODEL)
    r2, d2 = _quick_probe(SUPERVISOR_MODEL)
    print("  responded in %.1fs -> %r" % (d2, r2[:40]))
    print("  NOTE: both models stay resident (keep_alive), so VRAM must hold BOTH.")
    print("        30B (~18GB) + 7B (~5GB) needs ~23GB - fits 2x T4 (32GB).")
else:
    print("Supervisor shares the code model (self-critique).")
print("Model verification: PASSED")

## 7 · Benchmark helpers
Test-block wrapping, call-signature hints, and the HumanEval public tests (the worked examples in each docstring).

In [ ]:
import re, random, ast, doctest

# HumanEval signatures use typing names; provide them so a missing import is not
# scored as a logic failure (standard in HumanEval harnesses).
HUMANEVAL_SETUP = ("from typing import List, Tuple, Dict, Set, Optional, Any, Union\n"
                   "import math\n")


def extract_entry_point(test_list):
    for t in test_list:
        m = re.search(r'assert\s+([A-Za-z_]\w*)\s*\(', t)
        if m:
            return m.group(1)
        m = re.search(r'([A-Za-z_]\w*)\s*\(', t)
        if m:
            return m.group(1)
    return None


def wrap_composite_test(test_code, entry):
    """Turn a benchmark test BLOCK into one runnable unit.
      1. defines `def check(candidate)`  -> call check(entry)
      2. refers to a bare `candidate`    -> bind candidate = entry first
      3. plain assert statements         -> run as-is
    """
    t = test_code or ""
    if "def check(" in t:
        return "%s\n\ncheck(%s)\n" % (t, entry)
    if "candidate" in t:
        return "candidate = %s\n%s\n" % (entry, t)
    return t


def extract_call_signature(asserts, entry, max_len=160):
    """Hint showing HOW the function is called, taken from the example tests."""
    if not entry:
        return None
    for a in asserts:
        i = a.find(entry + "(")
        if i < 0:
            continue
        j = i + len(entry)
        depth, k = 0, j
        for k in range(j, len(a)):           # balanced-paren scan
            if a[k] == "(":
                depth += 1
            elif a[k] == ")":
                depth -= 1
                if depth == 0:
                    break
        call = a[i:k + 1]
        inner = call[len(entry) + 1:-1]
        depth, n_args = 0, (1 if inner.strip() else 0)
        for ch in inner:
            if ch in "([{":
                depth += 1
            elif ch in ")]}":
                depth -= 1
            elif ch == "," and depth == 0:
                n_args += 1
        if len(call) > max_len:
            call = call[:max_len] + "...)"
        return ("The function is called with %d positional argument(s). "
                "Example call from the tests: %s" % (n_args, call))
    return None


# ---- HumanEval public tests: the examples in the docstring ------------------
_ARROWS = ["==>", "➞", "=>", "->", "==", "should return", "returns", "="]

def _is_expr(src):
    try:
        ast.parse(src.strip(), mode="eval"); return True
    except Exception:
        return False

def _literal(src):
    try:
        ast.literal_eval(src.strip()); return True
    except Exception:
        return False

def humaneval_public_tests(prompt, entry):
    """Public tests = the worked examples a HumanEval prompt shows the model.
    Handles doctest blocks (>>> f(x) / expected) and one-line forms such as
    `f(x) ==> y`, `f(x) -> y`, `f(x) == y`. Only literal expected values are kept."""
    tests = []
    lines = prompt.splitlines()
    i = 0
    while i < len(lines):                     # doctest-style: ">>> call" then the value
        s = lines[i].strip()
        if s.startswith(">>>"):
            src = s[3:].strip()
            want, j = [], i + 1
            while j < len(lines) and lines[j].strip() and not lines[j].strip().startswith(">>>") \
                    and not lines[j].strip().startswith('"""'):
                want.append(lines[j].strip()); j += 1
            w = " ".join(want)
            if entry + "(" in src and w and _is_expr(src) and _literal(w):
                tests.append("assert (%s) == (%s)" % (src, w))
            i = j
        else:
            i += 1
    if tests:
        return tests
    for line in lines:
        s = line.strip().lstrip("#").strip()
        if not s.startswith(entry + "("):
            continue
        for arrow in _ARROWS:
            if arrow in s:
                left, right = s.split(arrow, 1)
                right = right.strip().rstrip(".").strip()
                right = {"true": "True", "false": "False"}.get(right, right)
                if _is_expr(left) and _literal(right):
                    tests.append("assert (%s) == (%s)" % (left.strip(), right))
                break
    return tests


def sample_main_set(problems, n, seed=SEED):
    """Reproducible sample: shuffle once with SEED, take the first n (None = all)."""
    rr = random.Random(seed)
    order = list(range(len(problems)))
    rr.shuffle(order)
    n_eff = len(problems) if n is None else min(n, len(problems))
    return [problems[i] for i in order[:n_eff]]

print("Benchmark helpers ready.")


## 8 · Execution sandbox
Every program runs in a separate process with a time and memory limit. `assert` mode for function tests, `stdin` mode for LiveCodeBench programs. Grading results are memoised so a shared draft is executed once.

In [ ]:
import tempfile, subprocess, os, json, shutil, sys, hashlib

# Runner executed inside the subprocess. Two modes:
#   "assert": exec the solution once, then run each test string in that namespace
#   "stdin" : each test is {"input", "output"}; the program runs once per test with
#             that stdin and its stdout is compared token by token (LiveCodeBench)
RUNNER_SRC = r'''
import json, sys, os, ast, io, traceback

def _describe(test_src, ns):
    # For 'assert <left> <op> <right>' return the actual vs expected values.
    try:
        node = ast.parse(test_src.strip()).body[0]
        if isinstance(node, ast.Assert) and isinstance(node.test, ast.Compare) \
           and len(node.test.ops) == 1:
            left = ast.Expression(node.test.left)
            right = ast.Expression(node.test.comparators[0])
            lval = eval(compile(left, '<l>', 'eval'), ns)
            rval = eval(compile(right, '<r>', 'eval'), ns)
            op = type(node.test.ops[0]).__name__
            return ('got %r, expected (%s) %r' % (lval, op, rval))[:600]
    except Exception:
        return None
    return None

def _tokens_equal(got, exp):
    g, e = got.split(), exp.split()
    if len(g) != len(e):
        return False
    for a, b in zip(g, e):
        if a == b:
            continue
        try:
            fa, fb = float(a), float(b)
            if abs(fa - fb) <= 1e-6 * max(1.0, abs(fb)):
                continue
        except ValueError:
            pass
        return False
    return True

def _clip(s, n=300):
    s = s if isinstance(s, str) else repr(s)
    return s if len(s) <= n else s[:n] + '...'

def run_assert(payload, out):
    ns = {}
    try:
        if payload.get('setup'):
            exec(compile(payload['setup'], '<setup>', 'exec'), ns)
        exec(compile(payload['code'], '<solution>', 'exec'), ns)
    except BaseException:
        out['import_ok'] = False
        out['fatal'] = traceback.format_exc()
        return
    for i, t in enumerate(payload['tests']):
        r = {'index': i, 'passed': False, 'error': None, 'test': t[:200], 'detail': None}
        try:
            exec(compile(t, '<test%d>' % i, 'exec'), ns)
            r['passed'] = True
        except AssertionError:
            r['error'] = traceback.format_exc()
            r['detail'] = _describe(t, ns)
        except BaseException:
            r['error'] = traceback.format_exc()
        out['results'].append(r)
        if payload.get('stop_on_fail') and not r['passed']:
            break

def run_stdin(payload, out):
    # Each case runs the program as its own process with the case on real stdin,
    # so input(), sys.stdin.buffer and open(0) all behave as on a judge.
    import subprocess
    try:
        compile(payload['code'], '<solution>', 'exec')
    except BaseException:
        out['import_ok'] = False
        out['fatal'] = traceback.format_exc()
        return
    with open('solution.py', 'w') as f:
        f.write(payload['code'])
    for i, t in enumerate(payload['tests']):
        r = {'index': i, 'passed': False, 'error': None,
             'test': 'stdin case %d' % i, 'detail': None}
        err, got = None, ''
        try:
            pr = subprocess.run([sys.executable, 'solution.py'], input=t['input'],
                                capture_output=True, text=True,
                                timeout=payload.get('case_timeout', 10))
            got = pr.stdout
            if pr.returncode != 0:
                err = (pr.stderr or 'exit code %d' % pr.returncode)[-1500:]
        except subprocess.TimeoutExpired:
            err = 'TimeoutExpired: case %d exceeded the per-case limit' % i
        if err:
            r['error'] = err
        elif _tokens_equal(got, t['output']):
            r['passed'] = True
        else:
            r['error'] = 'AssertionError: wrong answer'
            r['detail'] = 'input %s -> got %s, expected %s' % (
                _clip(t['input']), _clip(got), _clip(t['output']))
        out['results'].append(r)
        if payload.get('stop_on_fail') and not r['passed']:
            break

def main():
    d = sys.argv[1]
    with open(os.path.join(d, 'payload.json')) as f:
        payload = json.load(f)
    out = {'import_ok': True, 'fatal': None, 'results': []}
    (run_stdin if payload.get('mode') == 'stdin' else run_assert)(payload, out)
    sys.stdout.write('\n' + json.dumps(out) + '\n')

if __name__ == '__main__':
    main()
'''

def _rlimit_preexec(timeout, mem_gb):
    def _apply():
        try:
            import resource
            resource.setrlimit(resource.RLIMIT_CPU, (timeout + 1, timeout + 2))
            resource.setrlimit(resource.RLIMIT_AS, (mem_gb * 1024**3, mem_gb * 1024**3))
        except Exception:
            pass
    return _apply

def run_in_sandbox(code, tests, setup="", timeout=None, mode="assert",
                   stop_on_fail=False, mem_gb=2):
    if timeout is None:
        timeout = TIMEOUT_SECONDS
    d = tempfile.mkdtemp(prefix="mas_")
    empty = {"timed_out": False, "import_ok": False, "fatal": None,
             "results": [], "n_tests": len(tests), "n_passed": 0,
             "n_failed": len(tests), "stdout": "", "stderr": ""}
    try:
        with open(os.path.join(d, "runner.py"), "w") as f:
            f.write(RUNNER_SRC)
        with open(os.path.join(d, "payload.json"), "w") as f:
            json.dump({"code": code, "setup": setup, "tests": tests, "mode": mode,
                       "stop_on_fail": stop_on_fail}, f)
        kw = dict(capture_output=True, text=True, timeout=timeout, cwd=d,
                  stdin=subprocess.DEVNULL)
        if os.name == "posix":
            kw["preexec_fn"] = _rlimit_preexec(timeout, mem_gb)
        try:
            proc = subprocess.run([sys.executable, "runner.py", d], **kw)
        except subprocess.TimeoutExpired:
            r = dict(empty); r["timed_out"] = True; r["fatal"] = "TimeoutExpired"
            r["stderr"] = "Timeout after %ss" % timeout; return r
        parsed = None
        for line in reversed(proc.stdout.strip().splitlines()):
            line = line.strip()
            if line.startswith("{"):
                try:
                    parsed = json.loads(line); break
                except Exception:
                    pass
        if parsed is None:
            r = dict(empty); r["fatal"] = (proc.stderr or "no runner output")[:2000]
            r["stdout"] = proc.stdout[-2000:]; r["stderr"] = proc.stderr[-2000:]; return r
        n_pass = sum(1 for x in parsed["results"] if x["passed"])
        return {"timed_out": False, "import_ok": parsed["import_ok"],
                "fatal": parsed["fatal"], "results": parsed["results"],
                "n_tests": len(tests), "n_passed": n_pass,
                "n_failed": len(tests) - n_pass,
                "stdout": proc.stdout[-2000:], "stderr": proc.stderr[-2000:]}
    finally:
        shutil.rmtree(d, ignore_errors=True)


def all_passed(res):
    return bool(res["import_ok"]) and res["n_tests"] > 0 and res["n_failed"] == 0


# ---- problem-aware wrappers -------------------------------------------------
# A problem carries: test_list (HIDDEN grading suite), public_tests, test_mode,
# test_setup_code, timeout. Grading results are memoised by (code, suite) so the
# same draft is executed once, not once per system.
_GRADE_CACHE = {}

def run_suite(p, code, tests, stop_on_fail=False, mode=None):
    mode = mode or p.get("test_mode", "assert")
    key = hashlib.sha256(json.dumps([code, tests, mode, stop_on_fail, p["task_id"]],
                                    sort_keys=True, default=str).encode()).hexdigest()
    if key in _GRADE_CACHE:
        return _GRADE_CACHE[key]
    res = run_in_sandbox(code, tests, p.get("test_setup_code", ""),
                         timeout=p.get("timeout", TIMEOUT_SECONDS), mode=mode,
                         stop_on_fail=stop_on_fail, mem_gb=p.get("mem_gb", 2))
    _GRADE_CACHE[key] = res
    return res

def grade_hidden(p, code):
    """Pass/fail on the hidden grading suite (evaluation only, never shown to agents)."""
    return all_passed(run_suite(p, code, p["test_list"], stop_on_fail=True))

def passes_public(p, code):
    t = p.get("public_tests") or []
    return all_passed(run_suite(p, code, t)) if t else None


def first_error(res):
    if res.get("timed_out"):
        return "TimeoutExpired: execution exceeded the time limit"
    if res.get("fatal"):
        return res["fatal"][:1500]
    for x in res.get("results", []):
        if not x["passed"] and x.get("error"):
            # BARE mode: raw traceback only - reproduces the weak original setup
            if globals().get("FEEDBACK_MODE", "structured") == "bare":
                return x["error"][:1500]
            parts = []
            t = x.get("test") or ""
            # only echo short, single-assert tests; composite check() bodies are
            # noise (the traceback already pinpoints the failing line)
            if t and len(t) < 200 and t.count("assert") <= 1:
                parts.append("Failing test: " + t)
            if x.get("detail"):          # e.g. "got 8, expected (Eq) 6"
                parts.append("Observed: " + x["detail"])
            parts.append(x["error"][:1200])
            return "\n".join(parts)[:1800]
    return ""

def classify_failure(res):
    if res.get("timed_out"):
        return "Timeout"
    text = first_error(res)
    if not text:
        return "None"
    if "SyntaxError" in text or "IndentationError" in text:
        return "SyntaxError"
    if "ModuleNotFoundError" in text or "ImportError" in text:
        return "Missing import"
    if "AssertionError" in text:
        return "AssertionError"
    if "TypeError" in text and ("argument" in text or "positional" in text):
        return "Wrong function signature"
    if "NameError" in text:
        return "Wrong function signature"
    if "Error" in text or "Exception" in text:
        return "RuntimeError"
    return "Other"

# self-tests: assert mode, stdin mode, wrong answers in both
_t = run_in_sandbox("def add(a,b):\n    return a+b\n", ["assert add(2,3)==5"])
print("Sandbox (assert) passes correct code  :", all_passed(_t))
_t2 = run_in_sandbox("def add(a,b):\n    return a-b\n", ["assert add(2,3)==5"])
print("Sandbox (assert) catches wrong answer :", not all_passed(_t2), "->", classify_failure(_t2))
_prog = "a, b = map(int, input().split())\nprint(a + b)\n"
_t3 = run_in_sandbox(_prog, [{"input": "2 3\n", "output": "5\n"}], mode="stdin")
print("Sandbox (stdin)  passes correct code  :", all_passed(_t3))
_t4 = run_in_sandbox(_prog, [{"input": "2 3\n", "output": "6\n"}], mode="stdin")
print("Sandbox (stdin)  catches wrong answer :", not all_passed(_t4), "->", _t4["results"][0]["detail"])


## 9 · LLM interface

In [ ]:
import requests, time, hashlib

class OllamaLLM:
    """Thin Ollama client with a prompt cache. The cache is what lets all three systems
    share ONE first draft per question (same prompt -> same program), and a fixed seed
    makes that draft reproducible. Metering (section 14) replays the cost of a cached
    answer, so every system is still charged for the draft it uses."""
    def __init__(self, model, temperature=0.1, host=OLLAMA_HOST, use_cache=True,
                 num_predict=NUM_PREDICT, num_ctx=NUM_CTX, keep_alive=KEEP_ALIVE, seed=SEED):
        self.model = model
        self.temperature = temperature
        self.host = host
        self.use_cache = use_cache
        self.num_predict = num_predict
        self.num_ctx = num_ctx
        self.keep_alive = keep_alive
        self.seed = seed
        self.cache = {}
        self.actual_calls = 0   # real network calls (cache misses)

    def options(self):
        return {"temperature": self.temperature, "num_predict": self.num_predict,
                "num_ctx": self.num_ctx, "seed": self.seed}

    def chat(self, system, user):
        key = hashlib.sha256(
            ("|".join([self.model, str(self.temperature), system, user])).encode()
        ).hexdigest()
        if self.use_cache and key in self.cache:
            return self.cache[key], 0.0, True
        msgs = []
        if system:
            msgs.append({"role": "system", "content": system})
        msgs.append({"role": "user", "content": user})
        t0 = time.time()
        r = requests.post(self.host + "/api/chat", timeout=900, json={
            "model": self.model, "stream": False, "messages": msgs,
            "keep_alive": self.keep_alive, "options": self.options()})
        dt = time.time() - t0
        r.raise_for_status()
        content = r.json()["message"]["content"]
        self.actual_calls += 1
        if self.use_cache:
            self.cache[key] = content
        return content, dt, False

def extract_code(text):
    import re
    blocks = re.findall(r"```(?:python|py|Python)?\s*\n(.*?)```", text, re.DOTALL)
    if blocks:
        return max(blocks, key=len).strip()
    return text.strip()

def extract_asserts(text, entry):
    import re
    blocks = re.findall(r"```(?:python|py)?\s*\n(.*?)```", text, re.DOTALL)
    body = "\n".join(blocks) if blocks else text
    out = []
    for ln in body.splitlines():
        s = ln.strip()
        if s.startswith("assert") and (entry is None or entry in s):
            try:
                compile(s, "<t>", "exec")      # drop half-written asserts
            except SyntaxError:
                continue
            out.append(s)
    return out


## 10 · Agents
**Code agent** writes and repairs code · **Test agent** writes extra asserts (system `repair_tests` only) · **Supervisor** diagnoses failures.

In [ ]:
class CodeGenerationAgent:
    SYS = ("You are an expert Python programmer. Write correct, self-contained "
           "Python. Return ONLY one ```python code block with the required "
           "code (plus any imports it needs). No prose.")

    def __init__(self, llm):
        self.llm = llm

    def generate(self, text, entry, hint=None, instruction=None):
        extra = ("\n%s\nMatch that call signature exactly.\n" % hint) if hint else ""
        task = instruction or ("Implement the function named `%s`.\n"
                               "Return only the function definition (and needed imports) "
                               "in one python code block." % entry)
        prompt = "Problem:\n%s\n%s\n%s" % (text, extra, task)
        raw, dt, cached = self.llm.chat(self.SYS, prompt)
        return extract_code(raw), dt

    def repair(self, text, entry, prev_code, feedback, hint=None, instruction=None):
        task = instruction or ("Return a corrected, complete implementation of `%s` in one "
                               "python code block. Fix the specific issue; keep the same "
                               "function name and signature." % entry)
        prompt = ("Problem:\n%s\n\nYour previous solution FAILED its tests.\n\n"
                  "Previous code:\n```python\n%s\n```\n\n"
                  "Reviewer feedback:\n%s\n\n%s%s"
                  % (text, prev_code, feedback, task, ("\n" + hint) if hint else ""))
        raw, dt, cached = self.llm.chat(self.SYS, prompt)
        return extract_code(raw), dt


In [ ]:
class TestGenerationAgent:
    """Writes extra black-box asserts from the problem statement. They join the
    VISIBLE test set of the repair_tests system; they never touch grading."""
    SYS = ("You are a meticulous test engineer. Write black-box assert-based "
           "unit tests derived from the PROBLEM statement, not from any given "
           "implementation.")

    def __init__(self, llm):
        self.llm = llm

    def generate(self, text, entry, code, call_form=None, max_tests=5):
        call_form = call_form or ("%s(...)" % entry)
        prompt = ("Problem:\n%s\n\nFunction under test: `%s`, called as `%s`.\n"
                  "Write 3-5 additional `assert` statements (one per line, each on a "
                  "single line) that check correctness based on the problem, including "
                  "edge cases. Compute every expected value by hand from the problem "
                  "statement. Return them in one python code block."
                  % (text, entry, call_form))
        raw, dt, cached = self.llm.chat(self.SYS, prompt)
        tests = extract_asserts(raw, entry)
        return tests[:max_tests], dt


In [ ]:
class SupervisorAgent:
    SYS = ("You are a senior code reviewer. Given failing test output, diagnose "
           "the ROOT CAUSE in 2-4 sentences and give one concrete, specific "
           "instruction to fix it. Be precise and short.")

    def __init__(self, llm):
        self.llm = llm

    def review(self, text, code, error_excerpt):
        prompt = ("Problem:\n%s\n\nCode:\n```python\n%s\n```\n\n"
                  "Failing test output:\n%s\n\n"
                  "Give the root cause and a concrete fix instruction."
                  % (text, code, error_excerpt))
        raw, dt, cached = self.llm.chat(self.SYS, prompt)
        return raw.strip(), dt

## 11 · Repair orchestrator
Generate → [test agent] → run visible tests → (supervisor → repair → run) × up to `MAX_RETRIES`. Grading happens outside, on the hidden suite.

In [ ]:
class MultiAgentOrchestrator:
    """generate -> [test agent] -> run VISIBLE tests -> (supervisor -> repair -> run) x N.

    Visible tests = public tests (or the hidden suite when FEEDBACK_TESTS == "hidden")
    plus, for repair_tests, the test agent's asserts. The program the loop ends with is
    graded on the HIDDEN suite, which the agents never see in "public" mode."""

    def __init__(self, code_agent, test_agent, supervisor, max_retries=3,
                 use_test_agent=False, combine_supervisor_repair=False,
                 early_stop_no_progress=True, name="repair"):
        self.code_agent = code_agent
        self.test_agent = test_agent
        self.supervisor = supervisor
        self.max_retries = max_retries
        self.use_test_agent = use_test_agent
        self.combine = combine_supervisor_repair
        self.early_stop = early_stop_no_progress
        self.name = name

    def solve(self, p):
        text, entry = p["text"], p["entry_point"]
        hint = p.get("signature_hint") if USE_SIGNATURE_HINT else None
        rec = {"task_id": p["task_id"], "system": self.name, "iterations": 0,
               "executions": 0, "exec_seconds": 0.0, "stop_reason": ""}
        trace = {"task_id": p["task_id"], "system": self.name,
                 "generated_tests": [], "feedback": [], "repairs": []}

        # 1) first draft (identical prompt to the baseline -> served from the cache)
        draft, _ = self.code_agent.generate(text, entry, hint, p.get("instruction"))
        trace["initial_code"] = draft

        # 2) test agent (only for function-style problems)
        gtests = []
        if self.use_test_agent and p.get("ta_supported", True):
            gtests, _ = self.test_agent.generate(text, entry, draft, p.get("ta_call"),
                                                 TEST_AGENT_MAX_TESTS)
        trace["generated_tests"] = gtests

        base = p["public_tests"] if FEEDBACK_TESTS == "public" else p["test_list"]
        visible = list(base or []) + list(gtests)
        rec["n_visible_tests"] = len(visible)
        rec["n_generated_tests"] = len(gtests)

        def run_visible(src):
            t0 = time.time()
            res = run_suite(p, src, visible)
            rec["exec_seconds"] += time.time() - t0
            rec["executions"] += 1          # a pipeline execution: counts toward cost
            return res

        # 3) evaluate the draft on the visible tests
        if visible:
            res = run_visible(draft)
            vis_pass = all_passed(res)
            progress = res["n_passed"]
        else:                                # nothing to test against -> nothing to repair
            res, vis_pass, progress = None, True, 0
            rec["stop_reason"] = "no visible tests"
        rec["draft_visible_pass"] = vis_pass

        cur, last_err, best = draft, (first_error(res) if res else ""), progress
        can_measure_progress = len(visible) > 1

        # 4) repair loop
        while (not vis_pass) and rec["iterations"] < self.max_retries:
            if self.combine:
                fb = "Fix the code so all tests pass. Failing output:\n" + last_err
                trace["feedback"].append("[direct] " + last_err[:400])
            else:
                fb, _ = self.supervisor.review(text, cur, last_err)
                trace["feedback"].append(fb)
            cur, _ = self.code_agent.repair(text, entry, cur, fb, hint,
                                            p.get("repair_instruction"))
            trace["repairs"].append(cur)
            rec["iterations"] += 1
            res = run_visible(cur)
            vis_pass, progress = all_passed(res), res["n_passed"]
            last_err = first_error(res)
            if vis_pass:
                rec["stop_reason"] = "visible tests pass"
                break
            if self.early_stop and can_measure_progress and progress <= best:
                rec["stop_reason"] = "no progress"
                break
            best = max(best, progress)
        if not rec["stop_reason"]:
            rec["stop_reason"] = "draft passed" if rec["iterations"] == 0 else "max retries"

        rec["final_visible_pass"] = vis_pass
        rec["final_code_changed"] = cur != draft
        trace["final_code"] = cur
        return rec, trace, draft, cur, gtests


## 12 · Baseline, evaluation and checkpointing
Each question runs every system in turn (baseline first, so it pays for the shared draft), then the evaluation-only checks, then is appended to `rows.jsonl`. A killed session loses at most the question in flight.

In [ ]:
import json, os, time
import pandas as pd
from tqdm.auto import tqdm

class BaselineSystem:
    """One generation. No tests are run as part of the pipeline."""
    def __init__(self, code_agent):
        self.code_agent = code_agent
        self.name = "baseline"

    def solve(self, p):
        hint = p.get("signature_hint") if USE_SIGNATURE_HINT else None
        code, _ = self.code_agent.generate(p["text"], p["entry_point"], hint,
                                           p.get("instruction"))
        rec = {"task_id": p["task_id"], "system": "baseline", "iterations": 0,
               "executions": 0, "exec_seconds": 0.0, "stop_reason": "single sample",
               "n_visible_tests": 0, "n_generated_tests": 0,
               "draft_visible_pass": None, "final_visible_pass": None,
               "final_code_changed": False}
        trace = {"task_id": p["task_id"], "system": "baseline", "initial_code": code,
                 "final_code": code}
        return rec, trace, code, code, []


def build_systems(num_predict):
    """Fresh agents for one benchmark. All systems share ONE llm (and its cache)."""
    global llm, supervisor_llm, code_agent, test_agent, supervisor, SYSTEM_OBJS
    llm = OllamaLLM(MODEL_NAME, temperature=TEMPERATURE, num_predict=num_predict)
    supervisor_llm = (OllamaLLM(SUPERVISOR_MODEL, temperature=TEMPERATURE,
                                num_predict=num_predict)
                      if SUPERVISOR_MODEL and SUPERVISOR_MODEL != MODEL_NAME else llm)
    code_agent = CodeGenerationAgent(llm)
    test_agent = TestGenerationAgent(llm)
    supervisor = SupervisorAgent(supervisor_llm)
    common = dict(max_retries=MAX_RETRIES, combine_supervisor_repair=COMBINE_SUPERVISOR_REPAIR,
                  early_stop_no_progress=EARLY_STOP_NO_PROGRESS)
    SYSTEM_OBJS = {
        "baseline": BaselineSystem(code_agent),
        "repair": MultiAgentOrchestrator(code_agent, test_agent, supervisor,
                                         use_test_agent=False, name="repair", **common),
        "repair_tests": MultiAgentOrchestrator(code_agent, test_agent, supervisor,
                                               use_test_agent=True, name="repair_tests", **common),
    }
    return llm


def _evaluate(p, rec, draft, final, gtests):
    """Evaluation-only checks. Nothing here is charged to any system."""
    rec["draft_hidden_pass"] = grade_hidden(p, draft)
    rec["final_hidden_pass"] = grade_hidden(p, final)
    rec["draft_public_pass"] = passes_public(p, draft)
    rec["final_public_pass"] = passes_public(p, final)
    rec["failure_type"] = ("None" if rec["draft_hidden_pass"] else
                           classify_failure(run_suite(p, draft, p["test_list"], stop_on_fail=True)))
    # test-agent diagnostics
    rec["ta_n_valid"] = float("nan")
    rec["ta_flags_draft"] = None
    rec["ta_caught"] = None
    rec["ta_false_alarm"] = None
    if gtests:
        if p.get("reference_code"):
            rr = run_suite(p, p["reference_code"], gtests, mode="assert")
            rec["ta_n_valid"] = float(rr["n_passed"]) if rr["import_ok"] else 0.0
        flags = not all_passed(run_suite(p, draft, gtests, mode="assert"))
        rec["ta_flags_draft"] = flags
        # caught: public tests missed a real bug that the generated tests flagged
        rec["ta_caught"] = bool(flags and not rec["draft_hidden_pass"]
                                and rec["draft_public_pass"] is not False)
        # false alarm: generated tests reject a draft that is actually correct
        rec["ta_false_alarm"] = bool(flags and rec["draft_hidden_pass"])
    return rec


def _append_jsonl(path, records):
    # a session killed mid-write leaves a line without its newline; start a fresh line
    if os.path.exists(path) and os.path.getsize(path) > 0:
        with open(path, "rb") as f:
            f.seek(-1, os.SEEK_END)
            if f.read(1) != b"\n":
                with open(path, "a") as g:
                    g.write("\n")
    with open(path, "a") as f:
        for r in records:
            f.write(json.dumps(r, default=str) + "\n")
        f.flush()
        os.fsync(f.fileno())


def _read_jsonl(path):
    out = []
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    out.append(json.loads(line))
                except Exception:
                    pass        # a line truncated by a killed session is skipped
    return out


def evaluate_question(p, tag):
    """Run every system on one question; return rows, traces and the calls it made."""
    rows, traces = [], []
    i_start = len(CALL_LOG)
    for name in SYSTEMS:
        i0 = len(CALL_LOG)
        rec, trace, draft, final, gtests = SYSTEM_OBJS[name].solve(p)
        cost = _cost_slice(i0)
        _tag_calls(i0, p["task_id"], name)
        rec = _evaluate(p, rec, draft, final, gtests)
        rec.update(cost)
        rec["benchmark"] = tag
        rows.append(rec)
        traces.append(trace)
    return rows, traces, CALL_LOG[i_start:]

print("Systems:", ", ".join(SYSTEM_LABEL[s] for s in SYSTEMS),
      "| feedback tests:", FEEDBACK_TESTS)


## 13 · Cost meter — GPU-seconds and joules (NVML)

- **GPU-seconds** = server-reported compute time (model load excluded) × number of GPUs holding
  the model. A 30B sharded over two T4s holds both cards for the whole call.
- **Joules** come from the GPU's hardware energy counter (exact on a T4), falling back to
  integrated power sampling. **Above idle** subtracts the idle draw of the resident model, so it
  is the *marginal* energy a pipeline adds.

In [ ]:
import time, threading, os, json

class GPUMeter:
    """NVML sampler: GPU-seconds, joules (hardware counter preferred), utilisation."""

    def __init__(self, sample_hz=10):
        self.ok, self.handles, self.indices = False, [], []
        self.sample_interval = 1.0 / max(sample_hz, 1)
        self.use_energy_counter = False
        self._nvml = None
        try:
            import pynvml
            pynvml.nvmlInit()
            self._nvml = pynvml
            n = pynvml.nvmlDeviceGetCount()
            self.indices = list(range(n))
            self.handles = [pynvml.nvmlDeviceGetHandleByIndex(i) for i in self.indices]
            try:
                for h in self.handles:
                    pynvml.nvmlDeviceGetTotalEnergyConsumption(h)
                self.use_energy_counter = True
            except Exception:
                self.use_energy_counter = False
            self.ok = n > 0
        except Exception as e:
            print("[GPUMeter] NVML unavailable (%s). Energy will be NaN." % e)

    def device_info(self):
        out = []
        for i, h in zip(self.indices, self.handles):
            name = self._nvml.nvmlDeviceGetName(h)
            if isinstance(name, bytes):
                name = name.decode()
            mem = self._nvml.nvmlDeviceGetMemoryInfo(h)
            out.append((i, name, mem.total / 1e9, mem.used / 1e9))
        return out

    def _power_w(self):
        return sum(self._nvml.nvmlDeviceGetPowerUsage(h) for h in self.handles) / 1000.0

    def _energy_mj(self):
        return sum(self._nvml.nvmlDeviceGetTotalEnergyConsumption(h) for h in self.handles)

    def measure_idle_watts(self, seconds=5.0):
        if not self.ok:
            return float("nan")
        s, t_end = [], time.time() + seconds
        while time.time() < t_end:
            s.append(self._power_w())
            time.sleep(self.sample_interval)
        return sum(s) / len(s) if s else float("nan")

    def start(self):
        if not self.ok:
            return {"t0": time.perf_counter(), "ok": False}
        tok = {"t0": time.perf_counter(), "ok": True, "stop": threading.Event(), "s": []}
        if self.use_energy_counter:
            tok["e0"] = self._energy_mj()

        def loop():
            while not tok["stop"].is_set():
                try:
                    tok["s"].append((time.perf_counter(), self._power_w()))
                except Exception:
                    pass
                tok["stop"].wait(self.sample_interval)

        tok["th"] = threading.Thread(target=loop, daemon=True)
        tok["th"].start()
        return tok

    def stop(self, tok):
        wall = time.perf_counter() - tok["t0"]
        if not tok.get("ok"):
            return {"wall_seconds": wall, "joules": float("nan"),
                    "mean_power_w": float("nan")}
        tok["stop"].set()
        tok["th"].join(timeout=2.0)
        if self.use_energy_counter:
            joules = (self._energy_mj() - tok["e0"]) / 1000.0
        else:
            s, joules = tok["s"], 0.0
            for (t1, p1), (t2, p2) in zip(s, s[1:]):
                joules += 0.5 * (p1 + p2) * (t2 - t1)      # trapezoid rule
            if not s:
                joules = float("nan")
        powers = [p for _, p in tok["s"]]
        return {"wall_seconds": wall, "joules": joules,
                "mean_power_w": (sum(powers) / len(powers)) if powers else float("nan")}


M1_METER = GPUMeter()
print("NVML available      :", M1_METER.ok)
print("Hardware energy ctr :", M1_METER.use_energy_counter)
for i, name, tot, used in M1_METER.device_info():
    print("  GPU %d: %-28s %.1f GB total, %.1f GB used" % (i, name, tot, used))
if not M1_METER.ok:
    print("\n  Without NVML the energy axis is NaN. GPU-seconds, calls and tokens still work.")

In [ ]:
# --- how many GPUs actually hold the model? ---------------------------------
# GPU-seconds scale linearly with this, so a wrong value scales the headline cost
# metric by exactly that factor. Detect it from VRAM rather than assuming.
def detect_n_gpus(min_gb=1.0):
    if not M1_METER.ok:
        return 1
    n = sum(1 for _, _, _, used in M1_METER.device_info() if used > min_gb)
    return max(n, 1)

# --- idle baseline ----------------------------------------------------------
# Keep the machine quiet for a few seconds. The model may be resident in VRAM;
# that draw is exactly what we want to subtract.
print("Calibrating idle power (5 s) ...")
IDLE_WATTS = M1_METER.measure_idle_watts(5.0)
N_GPUS = detect_n_gpus()

M1_CONFIG = {
    "N_GPUS": N_GPUS,
    "IDLE_WATTS": IDLE_WATTS,
    "MODEL": MODEL_NAME,
}
print("Idle power  : %s" % ("%.1f W" % IDLE_WATTS if IDLE_WATTS == IDLE_WATTS else "n/a"))
print("GPUs holding the model: %d  (GPU-seconds are multiplied by this)" % N_GPUS)
if N_GPUS == 1 and M1_METER.ok and len(M1_METER.indices) > 1:
    print("  NOTE: %d GPUs present but only 1 holds weights. If the model is sharded,"
          % len(M1_METER.indices))
    print("        re-run this cell AFTER the first generation so VRAM is populated.")

## 14 · Metered model calls

Every agent goes through `OllamaLLM.chat`, so metering it captures every cost axis. Each call is tagged with its role. The systems share one first draft through the prompt cache; the meter **replays the original cost on a cache hit** so each system is charged for it, and flags the replay so per-call analyses can drop it.

In [ ]:
import hashlib, requests, contextlib

CALL_LOG = []                 # one record per model call, in order
_CURRENT_ROLE = ["generator"]

@contextlib.contextmanager
def _role(name):
    _CURRENT_ROLE.append(name)
    try:
        yield
    finally:
        _CURRENT_ROLE.pop()


def _metered_chat(self, system, user):
    """Drop-in replacement for OllamaLLM.chat that records cost per call."""
    key = hashlib.sha256(
        ("|".join([self.model, str(self.temperature), system, user])).encode()
    ).hexdigest()
    role = _CURRENT_ROLE[-1]

    if not hasattr(self, "cost_cache"):
        self.cost_cache = {}

    # Cache hit: replay the ORIGINAL measured cost so a shared generation is still
    # charged to every system that consumes it. The record is flagged cached=True so
    # per-call analyses (section 22) can drop the duplicate.
    if self.use_cache and key in self.cache:
        prior = self.cost_cache.get(key)
        if prior is not None:
            rec = dict(prior)
            rec.update({"role": role, "cached": True})
            CALL_LOG.append(rec)
        return self.cache[key], 0.0, True

    msgs = ([{"role": "system", "content": system}] if system else []) + \
           [{"role": "user", "content": user}]

    tok = M1_METER.start()
    try:
        r = requests.post(self.host + "/api/chat", timeout=900, json={
            "model": self.model, "stream": False, "messages": msgs,
            "keep_alive": self.keep_alive, "options": self.options()})
        r.raise_for_status()
        data = r.json()
    finally:
        m = M1_METER.stop(tok)

    content = data["message"]["content"]
    self.actual_calls += 1

    total_ns = data.get("total_duration")
    load_ns = data.get("load_duration", 0) or 0
    # Exclude one-off model load: it is amortised across the run, not a per-task cost.
    compute_s = (max(total_ns / 1e9 - load_ns / 1e9, 0.0) if total_ns
                 else m["wall_seconds"])
    joules = m["joules"]
    # A reload inside the window would put load energy into this call's joules but not
    # into its GPU-seconds. Scale it out and flag the call.
    load_s = load_ns / 1e9
    if load_s > 0.5 and m["wall_seconds"] > 0 and joules == joules:
        joules = joules * max(m["wall_seconds"] - load_s, 0.0) / m["wall_seconds"]

    cost = {
        "role": role,
        "cached": False,
        "gpu_seconds": compute_s * M1_CONFIG["N_GPUS"],
        "wall_seconds": m["wall_seconds"],
        "joules": joules,
        # power over the SAME window the energy was measured in (all cards)
        "mean_power_w": (m["joules"] / m["wall_seconds"]) if m["wall_seconds"] > 0 else float("nan"),
        "prompt_tokens": data.get("prompt_eval_count", 0) or 0,
        "completion_tokens": data.get("eval_count", 0) or 0,
        "load_seconds": load_s,
        "load_flag": load_s > 0.5,
    }
    if self.use_cache:
        self.cache[key] = content
        self.cost_cache[key] = cost
    CALL_LOG.append(dict(cost))
    return content, m["wall_seconds"], False


OllamaLLM.chat = _metered_chat
print("OllamaLLM.chat is now metered.")

# --- role tagging on the agent entry points ---------------------------------
def _tag_role(cls, method, role_name):
    orig = getattr(cls, method)
    if getattr(orig, "_m1_tagged", False):
        return
    def wrapper(self, *a, **kw):
        with _role(role_name):
            return orig(self, *a, **kw)
    wrapper._m1_tagged = True
    setattr(cls, method, wrapper)

_tag_role(CodeGenerationAgent, "generate", "generator")
_tag_role(CodeGenerationAgent, "repair",   "repair")
_tag_role(TestGenerationAgent, "generate", "test_writer")
_tag_role(SupervisorAgent,     "review",   "supervisor")
print("Role tagging installed: generator / repair / test_writer / supervisor")


# --- per-(question, system) cost --------------------------------------------
ROLE_KEYS = ["generator", "test_writer", "supervisor", "repair"]

def _cost_slice(i0):
    rows = CALL_LOG[i0:]
    out = {"gpu_seconds": 0.0, "wall_seconds": 0.0, "joules": 0.0,
           "prompt_tokens": 0, "completion_tokens": 0,
           "calls_total": 0, "calls_cached": 0, "joules_missing": 0}
    for k in ROLE_KEYS:
        out["calls_" + k] = 0
    for r in rows:
        out["calls_total"] += 1
        out["calls_" + r["role"]] = out.get("calls_" + r["role"], 0) + 1
        if r.get("cached"):
            out["calls_cached"] += 1
        out["gpu_seconds"] += r.get("gpu_seconds", 0.0)
        out["wall_seconds"] += r.get("wall_seconds", 0.0)
        out["prompt_tokens"] += r.get("prompt_tokens", 0)
        out["completion_tokens"] += r.get("completion_tokens", 0)
        j = r.get("joules", float("nan"))
        if j != j:
            out["joules_missing"] += 1
        else:
            out["joules"] += j
    out["total_tokens"] = out["prompt_tokens"] + out["completion_tokens"]
    # IDLE_WATTS is summed across ALL cards and the joules were measured over the
    # calls' wall windows, so the idle share is idle W x wall seconds.
    idle = M1_CONFIG["IDLE_WATTS"]
    out["joules_above_idle"] = (max(out["joules"] - idle * out["wall_seconds"], 0.0)
                                if (idle == idle and not out["joules_missing"])
                                else float("nan"))
    if out["joules_missing"]:
        out["joules"] = float("nan")
    out["mean_power_w"] = (out["joules"] / out["wall_seconds"]
                           if out["wall_seconds"] > 0 else float("nan"))
    return out


def _tag_calls(i0, task_id, system):
    """Stamp each call made during one solve() with its question and system."""
    for r in CALL_LOG[i0:]:
        r["task_id"] = task_id
        r["system"] = system

print("Per-question cost attribution ready.")


## 15 · Load MBPP+, HumanEval+ and LiveCodeBench
Public tests that the reference solution itself fails are dropped before the run.

In [ ]:
import json, re, base64, zlib, pickle
from datasets import load_dataset

M1_SOURCES = {
    ("mbpp", "plus"):       [("evalplus/mbppplus",)],
    ("humaneval", "plus"):  [("evalplus/humanevalplus",), ("evalplus/HumanEvalPlus",)],
}
M1_LABELS = {("mbpp", "plus"): "MBPP+", ("humaneval", "plus"): "HumanEval+",
             ("livecodebench", "lite"): "LiveCodeBench"}
M1_TAGS = {("mbpp", "plus"): "mbpp_plus", ("humaneval", "plus"): "humaneval_plus",
           ("livecodebench", "lite"): "livecodebench"}


def _validate_public(probs):
    """Drop public tests that the reference solution itself fails (a handful of MBPP
    asserts disagree with the MBPP+ corrected reference). Returns #tests dropped."""
    dropped = 0
    for p in probs:
        ref, pub = p.get("reference_code"), p.get("public_tests") or []
        if not ref or not pub:
            continue
        res = run_in_sandbox(ref, pub, p["test_setup_code"], timeout=p["timeout"])
        if res["import_ok"] and not res.get("timed_out"):
            keep = [pub[r["index"]] for r in res["results"] if r["passed"]]
            dropped += len(pub) - len(keep)
            p["public_tests"] = keep
    return dropped


def _load_evalplus(benchmark, variant):
    dsd, last = None, None
    for cfg in M1_SOURCES[(benchmark, variant)]:
        try:
            dsd = load_dataset(*cfg); break
        except Exception as e:
            last = e
    if dsd is None:
        raise RuntimeError("Failed to load %s: %s" % (M1_LABELS[(benchmark, variant)], last))
    split = "test" if "test" in dsd else list(dsd.keys())[0]
    d = dsd[split]
    probs = []
    if benchmark == "humaneval":
        for i in range(len(d)):
            r = d[i]
            ep = r.get("entry_point")
            prompt = r.get("prompt") or ""
            probs.append({
                "task_id": r.get("task_id"), "text": prompt,
                "reference_code": prompt + (r.get("canonical_solution") or ""),
                "test_list": [wrap_composite_test(r.get("test"), ep)],     # hidden
                "public_tests": humaneval_public_tests(prompt, ep),       # docstring examples
                "test_setup_code": HUMANEVAL_SETUP, "entry_point": ep,
                "test_mode": "assert", "timeout": TIMEOUT_SECONDS,
                "signature_hint": None,   # the prompt already carries the signature
                "ta_supported": True, "ta_call": "%s(...)" % ep,
            })
    else:
        tcol = "text" if "text" in d.column_names else "prompt"
        scol = ("test_setup_code" if "test_setup_code" in d.column_names
                else ("test_imports" if "test_imports" in d.column_names else None))
        for i in range(len(d)):
            r = d[i]
            setup = ""
            if scol:
                raw = r.get(scol, "") or ""
                setup = "\n".join(raw) if isinstance(raw, (list, tuple)) else str(raw)
            original = list(r.get("test_list") or [])
            ep = extract_entry_point(original)
            hidden = original
            ext = r.get("test") or r.get("assertion") or ""
            if isinstance(ext, (list, tuple)):
                ext = "\n".join(ext)
            if MBPP_PLUS_USE_EXTENDED_TESTS and ext and ext.strip() and ep:
                hidden = [wrap_composite_test(ext, ep)]
            probs.append({
                "task_id": r.get("task_id"), "text": r.get(tcol) or "",
                "reference_code": r.get("code", ""),
                "test_list": hidden, "public_tests": original,
                "test_setup_code": setup, "entry_point": ep,
                "test_mode": "assert", "timeout": TIMEOUT_SECONDS,
                "signature_hint": extract_call_signature(original, ep),
                "ta_supported": True, "ta_call": "%s(...)" % ep,
            })
    return [p for p in probs if p["entry_point"] and p["test_list"]]


# ---- LiveCodeBench ----------------------------------------------------------
LCB_SETUP = ("from typing import *\nfrom collections import *\nfrom itertools import *\n"
             "from functools import *\nfrom heapq import *\nfrom bisect import *\n"
             "import math, string, re, sys, random, collections, heapq, bisect, itertools, functools\n"
             "import json as _json\n"
             "sys.setrecursionlimit(10**6)\n"
             "def _lcb_norm(x):\n"
             "    if isinstance(x, tuple): x = list(x)\n"
             "    if isinstance(x, list): return [_lcb_norm(v) for v in x]\n"
             "    if isinstance(x, float): return round(x, 6)\n"
             "    return x\n")

LCB_STDIN_INSTR = ("Read the inputs from stdin, solve the problem and write the answer to stdout "
                   "(do not directly test on the sample inputs). Return one complete Python "
                   "program in one python code block.")
LCB_STDIN_REPAIR = ("Return a corrected, complete Python program in one python code block. "
                    "It must read from stdin and write to stdout.")


def _lcb_decode_private(s):
    if not s:
        return []
    try:
        return json.loads(s)
    except Exception:
        return json.loads(pickle.loads(zlib.decompress(base64.b64decode(s.encode("utf-8")))))


def _lcb_functional_asserts(cases, func):
    out = []
    for c in cases:
        args = [ln for ln in c["input"].split("\n") if ln.strip()]
        out.append("assert _lcb_norm(Solution().%s(*[_json.loads(_a) for _a in %r])) == "
                   "_lcb_norm(_json.loads(%r))" % (func, args, c["output"].strip()))
    return out


def _load_livecodebench():
    from huggingface_hub import HfApi, hf_hub_download
    files = LCB_FILES
    if not files:
        files = sorted(f for f in HfApi().list_repo_files(LCB_REPO, repo_type="dataset")
                       if re.fullmatch(r"test\d*\.jsonl", f))
    seen, raw = set(), []
    for fn in files:
        path = hf_hub_download(LCB_REPO, fn, repo_type="dataset")
        with open(path) as f:
            for line in f:
                if not line.strip():
                    continue
                r = json.loads(line)
                if r["question_id"] in seen:
                    continue
                seen.add(r["question_id"]); raw.append(r)
    print("  LiveCodeBench files: %s -> %d unique problems" % (", ".join(files), len(raw)))

    probs = []
    for r in raw:
        public = json.loads(r["public_test_cases"]) if r.get("public_test_cases") else []
        private = _lcb_decode_private(r.get("private_test_cases"))
        if not public and not private:
            continue
        meta = json.loads(r["metadata"]) if r.get("metadata") else {}
        ttype = (public or private)[0].get("testtype", "stdin")
        p = {"task_id": "LCB/%s" % r["question_id"], "text": r["question_content"],
             "contest_date": str(r.get("contest_date", ""))[:10],
             "difficulty": r.get("difficulty"), "platform": r.get("platform"),
             "reference_code": None, "test_setup_code": LCB_SETUP,
             "timeout": LCB_TIMEOUT_S, "mem_gb": 3, "signature_hint": None}
        if ttype == "functional":
            func = meta.get("func_name")
            if not func:
                continue
            starter = r.get("starter_code") or ""
            p.update({
                "entry_point": func, "test_mode": "assert",
                "public_tests": _lcb_functional_asserts(public, func),
                "test_list": _lcb_functional_asserts(public + private, func),
                "instruction": ("Complete the following starter code and return the whole "
                                "class in one python code block:\n```python\n%s\n```" % starter),
                "repair_instruction": ("Return the corrected, complete `Solution` class in one "
                                       "python code block; keep the method name `%s` and its "
                                       "signature." % func),
                "ta_supported": True, "ta_call": "Solution().%s(...)" % func,
            })
        else:
            clean = lambda cs: [{"input": c["input"], "output": c["output"]} for c in cs]
            p.update({
                "entry_point": "main program", "test_mode": "stdin",
                "public_tests": clean(public), "test_list": clean(public + private),
                "instruction": LCB_STDIN_INSTR, "repair_instruction": LCB_STDIN_REPAIR,
                "ta_supported": False,     # asserts cannot drive a stdin program
            })
        probs.append(p)

    window = [p for p in probs if p["contest_date"] >= LCB_MIN_DATE]
    if len(window) < M1_MIN_PROBLEMS:
        print("  only %d problems on/after %s -> taking the newest %d instead"
              % (len(window), LCB_MIN_DATE, M1_MIN_PROBLEMS))
        window = sorted(probs, key=lambda p: p["contest_date"], reverse=True)[:M1_MIN_PROBLEMS]
    if window:
        print("  contest dates %s .. %s  (check these against the model's training cutoff)"
              % (min(p["contest_date"] for p in window), max(p["contest_date"] for p in window)))
    return window


def prepare_benchmark_m1(benchmark, variant=None):
    """-> (problems, label, tag)"""
    key = (benchmark, variant)
    if benchmark == "livecodebench":
        probs = _load_livecodebench()
    elif key in M1_SOURCES:
        probs = _load_evalplus(benchmark, variant)
    else:
        raise ValueError("Unknown benchmark/variant: %r" % (key,))
    return probs, M1_LABELS[key], M1_TAGS[key]


BENCH_PROBLEMS = {}
for _k in M1_BENCHMARKS:
    try:
        _p, _l, _t = prepare_benchmark_m1(*_k)
        _drop = _validate_public(_p)
        BENCH_PROBLEMS[_t] = (_p, _l)
        n_pub = sum(1 for p in _p if p["public_tests"])
        n_ta = sum(1 for p in _p if p.get("ta_supported"))
        print("%-15s %-14s %4d problems | %d with public tests (%d reference-failing public tests dropped)"
              " | test agent usable on %d"
              % (_t, _l, len(_p), n_pub, _drop, n_ta))
    except Exception as e:
        print("%-15s FAILED: %s" % (_k, e))


## 16 · Run the experiment
The long cell. Safe to re-run: finished questions are skipped. If the session budget runs out, start a new session and run all cells again.

In [ ]:
import uuid

M1_RESULTS = {}      # tag -> {"df", "calls", "label", "outdir", "complete", ...}
_STOPPED_FOR_BUDGET = False


def _warm_up():
    """One unmetered call so a model (re)load never lands inside a measured call."""
    try:
        requests.post(OLLAMA_HOST + "/api/chat", timeout=900, json={
            "model": MODEL_NAME, "stream": False, "keep_alive": KEEP_ALIVE,
            "messages": [{"role": "user", "content": "Reply with OK"}],
            "options": {"temperature": 0.0, "num_predict": 4, "num_ctx": NUM_CTX}})
    except Exception as e:
        print("  warm-up failed (continuing):", e)


def _n_for(tag, available):
    n = M1_N_PROBLEMS.get(tag)
    n_eff = available if n is None else min(n, available)
    if n_eff < M1_MIN_PROBLEMS and n_eff < available and not ALLOW_SMALL_N:
        raise ValueError("%s: %d questions is below the %d floor. Raise M1_N_PROBLEMS['%s'] "
                         "or set ALLOW_SMALL_N = True for a smoke test."
                         % (tag, n_eff, M1_MIN_PROBLEMS, tag))
    if n_eff < M1_MIN_PROBLEMS:
        print("  NOTE: %s has only %d questions (floor %d)." % (tag, n_eff, M1_MIN_PROBLEMS))
    return n_eff


def load_saved(tag, label):
    """Load everything already on disk for one benchmark (all sessions)."""
    outdir = os.path.join(M1_OUTPUT_DIR, tag)
    rows = _read_jsonl(os.path.join(outdir, "rows.jsonl"))
    calls = _read_jsonl(os.path.join(outdir, "calls.jsonl"))
    df = pd.DataFrame(rows)
    if len(df):
        # A question counts only when ONE run of it has a row for every system (a killed
        # session can leave a partial run behind). If a question ran twice, the later
        # complete run wins, and only that run's calls are kept.
        df["_order"] = range(len(df))
        per_run = df.groupby(["task_id", "run_uid"]).agg(n=("system", "nunique"),
                                                         last=("_order", "max")).reset_index()
        per_run = per_run[per_run.n == len(SYSTEMS)].sort_values("last")
        keep = per_run.drop_duplicates("task_id", keep="last")
        uids = set(keep.run_uid)
        df = (df[df.run_uid.isin(uids)].drop_duplicates(["task_id", "system"], keep="last")
              .drop(columns="_order"))
        calls = [c for c in calls if c.get("run_uid") in uids]
    return df.reset_index(drop=True), calls, outdir


def run_m1_benchmark(tag):
    global TIMEOUT_SECONDS, _STOPPED_FOR_BUDGET
    problems, label = BENCH_PROBLEMS[tag]
    main_set = sample_main_set(problems, _n_for(tag, len(problems)))
    outdir = os.path.join(M1_OUTPUT_DIR, tag)
    os.makedirs(outdir, exist_ok=True)

    prev, _, _ = load_saved(tag, label)
    done = set(prev["task_id"]) if (RESUME and len(prev)) else set()
    if not RESUME:
        for fn in ["rows.jsonl", "calls.jsonl", "traces.jsonl"]:
            if os.path.exists(os.path.join(outdir, fn)):
                os.remove(os.path.join(outdir, fn))
    todo = [p for p in main_set if p["task_id"] not in done]
    print("\n=== %s === %d questions (of %d available): %d done, %d to run"
          % (label, len(main_set), len(problems), len(main_set) - len(todo), len(todo)))
    with open(os.path.join(outdir, "m1_config.json"), "w") as f:
        json.dump({**EXPERIMENT_CONFIG, **M1_CONFIG, "tag": tag, "label": label,
                   "n_target": len(main_set),
                   "task_ids": [p["task_id"] for p in main_set]}, f, indent=2, default=str)

    build_systems(NUM_PREDICT_LCB if tag == "livecodebench" else NUM_PREDICT)
    _warm_up()
    CALL_LOG.clear()
    t0 = time.time()
    for p in tqdm(todo, desc="Evaluating [%s]" % label):
        if (time.time() - T_SESSION_START) / 3600.0 > SESSION_BUDGET_HOURS:
            _STOPPED_FOR_BUDGET = True
            print("  Session budget of %.2f h reached - stopping. Re-run this notebook in a new "
                  "session to resume." % SESSION_BUDGET_HOURS)
            break
        try:
            rows, traces, calls = evaluate_question(p, tag)
        except requests.exceptions.RequestException as e:
            print("  %s: model call failed (%s) - skipped, will retry on resume" % (p["task_id"], e))
            continue
        uid = uuid.uuid4().hex             # ties this question's rows to its calls
        for r in rows + calls + traces:
            r["run_uid"] = uid
        # rows.jsonl is written LAST: it is the commit marker for the question
        _append_jsonl(os.path.join(outdir, "calls.jsonl"), calls)
        _append_jsonl(os.path.join(outdir, "traces.jsonl"), traces)
        _append_jsonl(os.path.join(outdir, "rows.jsonl"), rows)
        CALL_LOG.clear()           # already on disk
    mins = (time.time() - t0) / 60.0

    df, calls, _ = load_saved(tag, label)
    complete = len(df) and df.task_id.nunique() >= len(main_set)
    M1_RESULTS[tag] = {"df": df, "calls": calls, "label": label, "outdir": outdir,
                       "n_target": len(main_set), "complete": bool(complete),
                       "minutes_this_session": mins}
    df.to_csv(os.path.join(outdir, "results_long.csv"), index=False)
    if len(df):
        pv = df.pivot(index="task_id", columns="system", values="final_hidden_pass").mean()
        print("  pass@1 so far (%d questions): %s   [%.1f min this session]"
              % (df.task_id.nunique(),
                 "  ".join("%s %.1f%%" % (SYSTEM_LABEL[s], 100 * pv[s]) for s in SYSTEMS if s in pv),
                 mins))
    return df


for _k in M1_BENCHMARKS:
    _tag = M1_TAGS[_k]
    if _tag not in BENCH_PROBLEMS:
        continue
    if _STOPPED_FOR_BUDGET:
        # still load what earlier sessions saved so the analysis covers it
        _df, _calls, _od = load_saved(_tag, BENCH_PROBLEMS[_tag][1])
        if len(_df):
            M1_RESULTS[_tag] = {"df": _df, "calls": _calls, "label": BENCH_PROBLEMS[_tag][1],
                                "outdir": _od, "n_target": None, "complete": False,
                                "minutes_this_session": 0.0}
        continue
    run_m1_benchmark(_tag)

M1_RESULTS = {t: R for t, R in M1_RESULTS.items() if len(R["df"])}
print("\nRuns finished ->", M1_OUTPUT_DIR)
for t, R in M1_RESULTS.items():
    print("  %-15s %4d questions%s" % (R["label"], R["df"].task_id.nunique(),
                                     "" if R["complete"] else "  (PARTIAL - resume to finish)"))


## 17 · Metrics per benchmark and system
`false_accept_%` = programs the loop accepted (visible tests pass) that fail the hidden suite.

In [ ]:
import numpy as np
import pandas as pd
from math import sqrt

if not M1_RESULTS:
    raise RuntimeError("No finished questions yet - the analysis below needs at least one. "
                       "Run section 16 (it resumes where it stopped).")
BENCH_TAGS = list(M1_RESULTS.keys())
BENCH_LABELS = [M1_RESULTS[t]["label"] for t in BENCH_TAGS]
ALL = pd.concat([R["df"].assign(benchmark=t, bench_label=R["label"])
                 for t, R in M1_RESULTS.items()], ignore_index=True)
ALL["calls_plus_exec"] = ALL["calls_total"] + ALL["executions"]
for c in ["final_hidden_pass", "draft_hidden_pass"]:
    ALL[c] = ALL[c].astype(bool)


def wilson(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    ph = k / n
    den = 1 + z * z / n
    mid = (ph + z * z / (2 * n)) / den
    half = z * sqrt(ph * (1 - ph) / n + z * z / (4 * n * n)) / den
    return (100 * (mid - half), 100 * (mid + half))


def _frac(series):
    s = series.dropna()
    return float(s.astype(bool).mean()) if len(s) else float("nan")


rows = []
for tag in BENCH_TAGS:
    for s in SYSTEMS:
        d = ALL[(ALL.benchmark == tag) & (ALL.system == s)]
        n = len(d)
        if not n:
            continue
        solved = int(d.final_hidden_pass.sum())
        lo, hi = wilson(solved, n)
        vis = d["final_visible_pass"].dropna().astype(bool)
        accepted = d.loc[vis[vis].index, "final_hidden_pass"].astype(bool)
        rows.append({
            "benchmark": M1_RESULTS[tag]["label"], "tag": tag, "system": s, "n_tasks": n,
            "pass@1": 100.0 * solved / n, "ci_lo": lo, "ci_hi": hi, "n_solved": solved,
            "calls_per_task": d.calls_total.mean(),
            "calls_plus_exec_per_task": d.calls_plus_exec.mean(),
            "executions_per_task": d.executions.mean(),
            "tokens_per_task": d.total_tokens.mean(),
            "prompt_tokens_per_task": d.prompt_tokens.mean(),
            "completion_tokens_per_task": d.completion_tokens.mean(),
            "gpu_s_per_task": d.gpu_seconds.mean(),
            "joules_per_task": d.joules.mean(),
            "joules_above_idle_per_task": d.joules_above_idle.mean(),
            "gpu_s_per_solved": d.gpu_seconds.sum() / solved if solved else float("nan"),
            "joules_per_solved": d.joules.sum() / solved if solved else float("nan"),
            "entered_repair_%": 100.0 * (d.iterations > 0).mean(),
            "mean_iterations": d.iterations.mean(),
            # of the programs the loop ACCEPTED (visible tests pass), how many fail hidden
            "false_accept_%": (100.0 * (~accepted).mean() if len(accepted) else float("nan")),
            "calls_" + "generator": d.calls_generator.mean(),
            "calls_test_writer": d.calls_test_writer.mean(),
            "calls_supervisor": d.calls_supervisor.mean(),
            "calls_repair": d.calls_repair.mean(),
        })

M1_METRICS = pd.DataFrame(rows)
M1_METRICS.to_csv(os.path.join(M1_OUTPUT_DIR, "metrics.csv"), index=False)
pd.set_option("display.width", 240, "display.max_columns", 60)
M1_METRICS[["benchmark", "system", "n_tasks", "pass@1", "ci_lo", "ci_hi", "calls_per_task",
            "calls_plus_exec_per_task", "tokens_per_task", "gpu_s_per_task", "joules_per_task",
            "gpu_s_per_solved", "entered_repair_%", "false_accept_%"]].round(2)


## 18 · Paired statistics
Same questions, every pair of systems → exact McNemar on the discordant pairs, Holm-corrected across all comparisons.
- **Do-no-harm gate:** regressions must be 0 · **n ≥ 45** before a p-value means anything

In [ ]:
from math import comb

def mcnemar_exact(b01, b10):
    n = b01 + b10
    if n == 0:
        return 1.0
    k = min(b01, b10)
    return min(1.0, 2.0 * sum(comb(n, i) for i in range(k + 1)) * (0.5 ** n))


def holm(pvals):
    """Holm-Bonferroni adjusted p-values, same order as the input."""
    m = len(pvals)
    order = sorted(range(m), key=lambda i: pvals[i])
    adj, running = [0.0] * m, 0.0
    for rank, i in enumerate(order):
        running = max(running, min(1.0, (m - rank) * pvals[i]))
        adj[i] = running
    return adj


COMPARISONS = [("baseline", "repair"), ("baseline", "repair_tests"), ("repair", "repair_tests")]
COMPARISONS = [(a, b) for a, b in COMPARISONS if a in SYSTEMS and b in SYSTEMS]

srows = []
for tag in BENCH_TAGS:
    wide = ALL[ALL.benchmark == tag].pivot(index="task_id", columns="system",
                                          values="final_hidden_pass").dropna()
    for a_sys, b_sys in COMPARISONS:
        a = wide[a_sys].astype(bool); b = wide[b_sys].astype(bool)
        b11 = int((a & b).sum()); b10 = int((a & ~b).sum())
        b01 = int((~a & b).sum()); b00 = int((~a & ~b).sum())
        n = len(wide)
        srows.append({"benchmark": M1_RESULTS[tag]["label"], "tag": tag,
                      "comparison": "%s -> %s" % (SYSTEM_LABEL[a_sys], SYSTEM_LABEL[b_sys]),
                      "a": a_sys, "b": b_sys, "n": n,
                      "a_pass": b11 + b10, "b_pass": b11 + b01,
                      "gains": b01, "regressions": b10, "b11": b11, "b00": b00,
                      "delta_pp": 100.0 * (b01 - b10) / n if n else float("nan"),
                      "p_exact": mcnemar_exact(b01, b10),
                      "do_no_harm_pass": b10 == 0, "powered_n45": n >= 45})

M1_STATS = pd.DataFrame(srows)
M1_STATS["p_holm"] = holm(list(M1_STATS["p_exact"])) if len(M1_STATS) else []
M1_STATS.to_csv(os.path.join(M1_OUTPUT_DIR, "stats.csv"), index=False)

print("Exact McNemar on the hidden-suite outcome; Holm correction across all %d tests.\n"
      % len(M1_STATS))
for r in M1_STATS.to_dict("records"):
    print("%-14s %-38s %3d/%-3d -> %3d/%-3d  %+5.1f pp  gains=%3d  regressions=%3d  "
          "p=%.4f  p_holm=%.4f  %s%s"
          % (r["benchmark"], r["comparison"], r["a_pass"], r["n"], r["b_pass"], r["n"],
             r["delta_pp"], r["gains"], r["regressions"], r["p_exact"], r["p_holm"],
             "significant" if r["p_holm"] < 0.05 else "n.s.",
             "" if r["do_no_harm_pass"] else "   [do-no-harm FAILED]"))
M1_STATS.round(4)


## 19 · Test agent diagnostics
Is a generated test correct (the reference solution passes it)? Did it catch a bug the public tests missed, or raise a false alarm on a correct draft?

In [ ]:
ta_rows = []
for tag in BENCH_TAGS:
    d = ALL[(ALL.benchmark == tag) & (ALL.system == "repair_tests")]
    if not len(d):
        continue
    used = d[d.n_generated_tests > 0]
    ref = used[used.ta_n_valid.notna()]
    n_tests = int(ref.n_generated_tests.sum())
    ta_rows.append({
        "benchmark": M1_RESULTS[tag]["label"], "questions": len(d),
        "questions_with_generated_tests": len(used),
        "tests_per_question": used.n_generated_tests.mean() if len(used) else float("nan"),
        # validity: a generated test the REFERENCE solution passes is a correct test
        "tests_checked_vs_reference": n_tests,
        "valid_tests_%": 100.0 * ref.ta_n_valid.sum() / n_tests if n_tests else float("nan"),
        "flagged_draft": int(used.ta_flags_draft.fillna(False).astype(bool).sum()),
        "caught_real_bug": int(used.ta_caught.fillna(False).astype(bool).sum()),
        "false_alarm_on_correct_draft": int(used.ta_false_alarm.fillna(False).astype(bool).sum()),
        "test_writer_calls_per_q": d.calls_test_writer.mean(),
        "extra_gpu_s_vs_repair": (d.gpu_seconds.mean() -
                                  ALL[(ALL.benchmark == tag) & (ALL.system == "repair")].gpu_seconds.mean()),
        "extra_joules_vs_repair": (d.joules.mean() -
                                   ALL[(ALL.benchmark == tag) & (ALL.system == "repair")].joules.mean()),
    })
TA_TABLE = pd.DataFrame(ta_rows)
TA_TABLE.to_csv(os.path.join(M1_OUTPUT_DIR, "test_agent_diagnostics.csv"), index=False)

for r in TA_TABLE.to_dict("records"):
    print("%s: test agent wrote tests for %d/%d questions (%.1f per question)"
          % (r["benchmark"], r["questions_with_generated_tests"], r["questions"],
             r["tests_per_question"]))
    if r["tests_checked_vs_reference"]:
        print("   %.0f%% of %d generated tests are correct (the reference solution passes them)"
              % (r["valid_tests_%"], r["tests_checked_vs_reference"]))
    else:
        print("   no reference solution -> test validity not measurable on this benchmark")
    print("   flagged the first draft on %d questions: %d real bugs the public tests missed, "
          "%d false alarms on a correct draft"
          % (r["flagged_draft"], r["caught_real_bug"], r["false_alarm_on_correct_draft"]))
    print("   extra cost vs repair: %+.2f GPU-s and %+.0f J per question"
          % (r["extra_gpu_s_vs_repair"], r["extra_joules_vs_repair"]))
TA_TABLE.round(2)


### 19b · Regressions and the keep-the-best rule
Why a system lost questions the baseline solved, and what pass@1 would be if the loop returned its best program by visible tests instead of its last one. Re-scored from saved traces: no model calls, same cost.

In [ ]:
# --- Why did a system lose questions the baseline solved? + "keep the best candidate" (no GPU) ---
# Part 1 lists every regression (baseline passes, system fails) and how the loop ended.
# Part 2 re-scores the saved traces: instead of returning the LAST program the loop wrote,
# return the one that passed the most visible tests (ties -> the earliest, so an unimproved
# draft is kept). Same model calls, same cost - only the final choice changes.
reg_rows, best_rows = [], []
for tag, R in M1_RESULTS.items():
    df = R["df"]
    w = df.pivot(index="task_id", columns="system", values="final_hidden_pass").astype(bool)
    for arm in [s for s in SYSTEMS if s != "baseline"]:
        broke = w[w["baseline"] & ~w[arm]].index
        sub = df[(df.system == arm) & df.task_id.isin(broke)]
        for r in sub.to_dict("records"):
            reg_rows.append({"benchmark": R["label"], "system": arm, "task_id": r["task_id"],
                             "draft_visible_pass": r["draft_visible_pass"],
                             "ta_false_alarm": r.get("ta_false_alarm"),
                             "iterations": r["iterations"], "stop_reason": r["stop_reason"]})

    probs = {p["task_id"]: p for p in BENCH_PROBLEMS[tag][0]}
    uids = set(df["run_uid"])
    for t in _read_jsonl(os.path.join(R["outdir"], "traces.jsonl")):
        if t.get("run_uid") not in uids or t["system"] == "baseline":
            continue
        p = probs[t["task_id"]]
        base = p["public_tests"] if FEEDBACK_TESTS == "public" else p["test_list"]
        visible = list(base or []) + list(t.get("generated_tests") or [])
        cands = [t["initial_code"]] + list(t.get("repairs") or [])
        if visible and len(cands) > 1:
            score = []
            for c in cands:
                res = run_suite(p, c, visible)
                score.append(res["n_passed"] if res["import_ok"] and not res["timed_out"] else -1)
            best = cands[max(range(len(cands)), key=lambda i: (score[i], -i))]
        else:
            best = cands[-1]
        best_rows.append({"benchmark": R["label"], "system": t["system"], "task_id": t["task_id"],
                          "last_pass": grade_hidden(p, cands[-1]), "best_pass": grade_hidden(p, best)})

REGRESSIONS = pd.DataFrame(reg_rows)
BEST = pd.DataFrame(best_rows)
REGRESSIONS.to_csv(os.path.join(M1_OUTPUT_DIR, "regressions.csv"), index=False)
BEST.to_csv(os.path.join(M1_OUTPUT_DIR, "keep_best_rescore.csv"), index=False)

if len(REGRESSIONS):
    print("Regressions vs baseline, by how the loop ended:")
    print(REGRESSIONS.groupby(["benchmark", "system", "stop_reason"]).size().to_string(), "\n")
    print("...of which the test agent raised a false alarm on the (correct) draft:")
    print(REGRESSIONS.groupby(["benchmark", "system"])["ta_false_alarm"]
          .apply(lambda s: int(s.fillna(False).astype(bool).sum())).to_string(), "\n")
print("pass@1 (%) - return the last program vs the best program by visible tests:")
print((BEST.groupby(["benchmark", "system"])[["last_pass", "best_pass"]].mean() * 100).round(1).to_string())


## 20 · Charts
Blue is always the baseline, orange repair, aqua (hatched) repair + test agent. Every figure is saved as a PNG.

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from matplotlib.patches import Patch

# Categorical slots 1-3 (validated all-pairs for colour-vision deficiency): one colour per
# SYSTEM, used identically in every figure.
SYS_COLOR = {"baseline": "#2a78d6", "repair": "#eb6834", "repair_tests": "#1baf7a"}
# Roles get their own validated set so a role is never mistaken for a system.
ROLE_COLOR = {"generator": "#2a78d6", "test_writer": "#4a3aa7",
              "supervisor": "#e87ba4", "repair": "#eda100"}
ROLE_LABEL = {"generator": "code agent - first draft", "test_writer": "test agent",
              "supervisor": "supervisor", "repair": "code agent - repair"}
C_GOOD, C_CRITICAL = "#0ca30c", "#d03b3b"      # reserved status colours
INK, INK_2, GRID = "#0b0b0b", "#52514e", "#d9d8d4"
SYS_HATCH = {"baseline": None, "repair": None, "repair_tests": "///"}   # secondary encoding

mpl.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 200, "savefig.bbox": "tight",
    "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
    "axes.edgecolor": GRID, "axes.labelcolor": INK_2, "text.color": INK,
    "xtick.color": INK_2, "ytick.color": INK_2,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True,
    "grid.color": GRID, "grid.linewidth": 0.6, "legend.frameon": False,
    "figure.facecolor": "white", "axes.facecolor": "white", "hatch.color": "white",
})

SYS_HANDLES = [Patch(facecolor=SYS_COLOR[s], hatch=SYS_HATCH[s], edgecolor="white",
                     label=SYSTEM_LABEL[s]) for s in SYSTEMS]
N_SYS = len(SYSTEMS)
BAR_W = 0.8 / N_SYS
BENCH_TICKS = ["%s\n(n = %d)" % (M1_RESULTS[t]["label"], M1_RESULTS[t]["df"].task_id.nunique())
               for t in BENCH_TAGS]


def mget(tag, system, col):
    r = M1_METRICS[(M1_METRICS.tag == tag) & (M1_METRICS.system == system)]
    return float(r[col].iloc[0]) if len(r) else float("nan")


def label_bars(ax, bars, fmt="{:.1f}", values=None, inside=False):
    top = ax.get_ylim()[1]
    for i, b in enumerate(bars):
        h = b.get_height() if values is None else values[i]
        if h != h:
            continue
        y_top = b.get_y() + b.get_height()
        if inside and b.get_height() > 0.12 * top:     # at the base: clear of the error bar
            ax.text(b.get_x() + b.get_width() / 2, b.get_y() + 0.025 * top, fmt.format(h),
                    ha="center", va="bottom", fontsize=8, color="white", fontweight="bold")
        else:
            ax.text(b.get_x() + b.get_width() / 2, y_top + 0.015 * top, fmt.format(h),
                    ha="center", va="bottom", fontsize=8, color=INK_2)


def grouped(ax, col, title, fmt="{:.1f}", ylabel=None, ci=False):
    x = np.arange(len(BENCH_TAGS))
    tops = []
    for k, s in enumerate(SYSTEMS):
        off = (k - (N_SYS - 1) / 2) * BAR_W
        vals = [mget(t, s, col) for t in BENCH_TAGS]
        kw = {}
        if ci:
            lo = [v - mget(t, s, "ci_lo") for v, t in zip(vals, BENCH_TAGS)]
            hi = [mget(t, s, "ci_hi") - v for v, t in zip(vals, BENCH_TAGS)]
            kw = dict(yerr=[lo, hi], error_kw=dict(ecolor=INK_2, elinewidth=1, capsize=3))
            tops += [mget(t, s, "ci_hi") for t in BENCH_TAGS]
        bars = ax.bar(x + off, vals, width=BAR_W * 0.92, color=SYS_COLOR[s],
                      hatch=SYS_HATCH[s], edgecolor="white", linewidth=0.6, **kw)
        tops += vals
        bars._m1_vals = vals
    finite = [v for v in tops if v == v]
    ax.set_ylim(0, (max(finite) if finite else 1) * 1.18)
    for c in ax.containers:
        if hasattr(c, "_m1_vals"):
            label_bars(ax, c, fmt, c._m1_vals if ci else None, inside=ci)
    ax.set_xticks(x); ax.set_xticklabels(BENCH_TICKS)
    if ylabel:
        ax.set_ylabel(ylabel)
    ax.set_title(title, color=INK, loc="left")
    ax.grid(axis="x", visible=False)
    return ax


def save(fig, name):
    fig.savefig(os.path.join(M1_OUTPUT_DIR, name))
    plt.show()

print("Chart style ready. Benchmarks:", BENCH_LABELS, "| systems:", SYSTEMS)


In [ ]:
# --- Figure 1 : correctness on the hidden suite, with 95% Wilson intervals ----
fig, ax = plt.subplots(figsize=(3.2 + 2.6 * len(BENCH_TAGS), 4.6))
grouped(ax, "pass@1", "Functional correctness (hidden tests)", "{:.1f}", "pass@1 (%)", ci=True)
ax.set_ylim(0, 105)
fig.legend(handles=SYS_HANDLES, loc="upper right", bbox_to_anchor=(0.99, 1.08), ncol=N_SYS)
fig.suptitle("pass@1 - %s" % MODEL_NAME, x=0.07, ha="left", fontsize=12.5, color=INK, y=1.08)
fig.text(0.07, 1.005, "Whiskers: 95%% Wilson interval. Repair feedback: %s tests." % FEEDBACK_TESTS,
         ha="left", fontsize=9, color=INK_2)
save(fig, "fig_01_pass_at_1.png")


In [ ]:
# --- Figure 2 : the five cost units, per question -----------------------------
panels = [("calls_per_task",           "Model calls / question",           "{:.2f}"),
          ("calls_plus_exec_per_task", "Calls + test runs / question",     "{:.2f}"),
          ("tokens_per_task",          "Tokens / question",                "{:.0f}"),
          ("gpu_s_per_task",           "GPU-seconds / question",           "{:.1f}"),
          ("joules_per_task",          "Joules / question (GPU cards)",    "{:.0f}")]
fig, axes = plt.subplots(1, len(panels), figsize=(4.1 * len(panels), 4.4))
for ax, (col, title, fmt) in zip(axes, panels):
    grouped(ax, col, title, fmt)
    ax.tick_params(axis="x", labelsize=8)
fig.legend(handles=SYS_HANDLES, loc="upper right", bbox_to_anchor=(0.995, 1.07), ncol=N_SYS)
fig.suptitle("What each system costs, per question, in five units", x=0.04, ha="left",
             fontsize=12.5, color=INK, y=1.07)
fig.text(0.04, 1.0, "Same model, same first draft. The baseline runs no tests, so its "
         "calls + test runs equals its calls.", ha="left", fontsize=9, color=INK_2)
fig.subplots_adjust(wspace=0.3)
save(fig, "fig_02_cost_units.png")


In [ ]:
# --- Figure 3 : what one solved question costs -------------------------------
fig, axes = plt.subplots(1, 2, figsize=(4 + 3.4 * len(BENCH_TAGS), 4.4))
grouped(axes[0], "gpu_s_per_solved", "GPU-seconds per solved question", "{:.1f}")
grouped(axes[1], "joules_per_solved", "Joules per solved question", "{:.0f}")
fig.legend(handles=SYS_HANDLES, loc="upper right", bbox_to_anchor=(0.99, 1.1), ncol=N_SYS)
fig.suptitle("Cost per solved question", x=0.05, ha="left", fontsize=12.5, color=INK, y=1.1)
fig.text(0.05, 1.025, "Total cost divided by questions solved. Lower is a better exchange rate; "
         "this is not a matched-budget comparison.", ha="left", fontsize=9, color=INK_2)
fig.subplots_adjust(wspace=0.22)
save(fig, "fig_03_cost_per_solved.png")


In [ ]:
# --- Figure 4 : accuracy-cost frontier, one panel per benchmark ---------------
fig, axes = plt.subplots(1, len(BENCH_TAGS), figsize=(4.8 * len(BENCH_TAGS), 4.4), squeeze=False)
for ax, tag in zip(axes[0], BENCH_TAGS):
    xs = [mget(tag, s, "gpu_s_per_task") for s in SYSTEMS]
    ys = [mget(tag, s, "pass@1") for s in SYSTEMS]
    lo = [y - mget(tag, s, "ci_lo") for y, s in zip(ys, SYSTEMS)]
    hi = [mget(tag, s, "ci_hi") - y for y, s in zip(ys, SYSTEMS)]
    for s, x, y, l, h in zip(SYSTEMS, xs, ys, lo, hi):
        if s != "baseline":
            ax.plot([xs[0], x], [ys[0], y], "-", color=GRID, linewidth=2, zorder=1)
        ax.errorbar([x], [y], yerr=[[l], [h]], fmt="none", ecolor=INK_2, elinewidth=1,
                    capsize=3, zorder=2)
        ax.scatter([x], [y], s=110, color=SYS_COLOR[s], zorder=3, edgecolor="white",
                   linewidth=2, marker="o" if s != "repair_tests" else "D")
        if s != "baseline":
            ax.annotate("%+.1f pp\n%+.1f GPU-s" % (y - ys[0], x - xs[0]), (x, y),
                        textcoords="offset points", xytext=(8, -4), ha="left", va="top",
                        fontsize=8, color=INK_2)
    ax.set_xlim(0, max(xs) * 1.35)
    ax.set_title("%s (n = %d)" % (M1_RESULTS[tag]["label"], M1_RESULTS[tag]["df"].task_id.nunique()),
                 color=INK, loc="left")
    ax.set_xlabel("GPU-seconds per question")
    lows = [mget(tag, q, "ci_lo") for q in SYSTEMS]
    ax.set_ylim(max(0, min(lows) - 8), 105)
axes[0][0].set_ylabel("pass@1 (%)")
handles = [plt.Line2D([], [], marker="o" if s != "repair_tests" else "D", linestyle="none",
                      markerfacecolor=SYS_COLOR[s], markeredgecolor="white", markersize=9,
                      label=SYSTEM_LABEL[s]) for s in SYSTEMS]
fig.legend(handles=handles, loc="upper right", bbox_to_anchor=(0.995, 1.08), ncol=N_SYS)
fig.suptitle("Accuracy against cost", x=0.04, ha="left", fontsize=12.5, color=INK, y=1.08)
fig.text(0.04, 1.005, "Up is better, left is cheaper. Labels give each system's change against "
         "the baseline; whiskers are 95% Wilson intervals.", ha="left", fontsize=9, color=INK_2)
fig.subplots_adjust(wspace=0.25)
save(fig, "fig_04_frontier.png")


In [ ]:
# --- Figure 5 : paired outcomes (gains / regressions) per comparison ----------
comps = ["%s -> %s" % (SYSTEM_LABEL[a], SYSTEM_LABEL[b]) for a, b in COMPARISONS]
SHORT = {"baseline": "base", "repair": "rep", "repair_tests": "rep+T"}
short_comps = ["%s -> %s" % (SHORT[a], SHORT[b]) for a, b in COMPARISONS]
fig, axes = plt.subplots(1, len(BENCH_TAGS), figsize=(4.9 * len(BENCH_TAGS), 4.6),
                         squeeze=False, sharey=True)
ymax = max(1, int(M1_STATS[["gains", "regressions"]].values.max()))
for ax, tag in zip(axes[0], BENCH_TAGS):
    st = M1_STATS[M1_STATS.tag == tag].set_index("comparison").reindex(comps)
    x = np.arange(len(comps))
    g = ax.bar(x - 0.19, st["gains"], width=0.36, color=C_GOOD, linewidth=0)
    r = ax.bar(x + 0.19, st["regressions"], width=0.36, color=C_CRITICAL, linewidth=0)
    ax.set_ylim(0, ymax * 1.45)
    label_bars(ax, g, "{:.0f}"); label_bars(ax, r, "{:.0f}")
    for xi, (_, row) in zip(x, st.iterrows()):
        ax.text(xi, ymax * 1.4, "p_holm = %.3f\n%s" % (row["p_holm"],
                "no harm" if row["do_no_harm_pass"] else "harm: %d" % row["regressions"]),
                ha="center", va="top", fontsize=7.5,
                color=INK_2 if row["do_no_harm_pass"] else C_CRITICAL)
    ax.set_xticks(x)
    ax.set_xticklabels(short_comps, fontsize=8.5)
    ax.set_title("%s (n = %d)" % (M1_RESULTS[tag]["label"], int(st["n"].iloc[0])),
                 color=INK, loc="left")
    ax.grid(axis="x", visible=False)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
axes[0][0].set_ylabel("questions")
fig.legend(handles=[Patch(facecolor=C_GOOD, label="gained (A fails, B passes)"),
                    Patch(facecolor=C_CRITICAL, label="regressed (A passes, B fails)")],
           loc="upper right", bbox_to_anchor=(0.995, 1.08), ncol=2)
fig.suptitle("Paired outcomes on the hidden suite", x=0.04, ha="left", fontsize=12.5,
             color=INK, y=1.08)
fig.text(0.04, 1.005, "A -> B on the same questions. base = baseline, rep = repair, rep+T = repair + "
         "test agent. Exact McNemar, Holm-corrected.", ha="left", fontsize=9, color=INK_2)
fig.subplots_adjust(wspace=0.12)
save(fig, "fig_05_paired_outcomes.png")


In [ ]:
# --- Figure 6 : how many repair rounds each question used ---------------------
arms = [s for s in SYSTEMS if s != "baseline"]
fig, axes = plt.subplots(1, len(BENCH_TAGS), figsize=(4.6 * len(BENCH_TAGS), 4.2),
                         squeeze=False, sharey=True)
rounds = list(range(MAX_RETRIES + 1))
w = 0.8 / max(len(arms), 1)
for ax, tag in zip(axes[0], BENCH_TAGS):
    for k, s in enumerate(arms):
        it = ALL[(ALL.benchmark == tag) & (ALL.system == s)].iterations
        share = [100.0 * (it == r).mean() for r in rounds]
        bars = ax.bar(np.array(rounds) + (k - (len(arms) - 1) / 2) * w, share, width=w * 0.92,
                      color=SYS_COLOR[s], hatch=SYS_HATCH[s], edgecolor="white", linewidth=0.6)
    ax.set_ylim(0, 115)
    for c in ax.containers:
        label_bars(ax, c, "{:.0f}")
    ax.set_xticks(rounds)
    ax.set_xlabel("repair rounds used")
    ax.set_title(M1_RESULTS[tag]["label"], color=INK, loc="left")
    ax.grid(axis="x", visible=False)
axes[0][0].set_ylabel("% of questions")
fig.legend(handles=[h for h, s in zip(SYS_HANDLES, SYSTEMS) if s != "baseline"],
           loc="upper right", bbox_to_anchor=(0.995, 1.08), ncol=len(arms))
fig.suptitle("Repair rounds per question", x=0.04, ha="left", fontsize=12.5, color=INK, y=1.08)
fig.text(0.04, 1.005, "0 = the draft passed its visible tests. Each round costs a supervisor "
         "call and a repair call.", ha="left", fontsize=9, color=INK_2)
fig.subplots_adjust(wspace=0.1)
save(fig, "fig_06_repair_rounds.png")

# how each loop ended (table)
STOP_TABLE = (ALL[ALL.system != "baseline"]
              .groupby(["bench_label", "system", "stop_reason"]).size()
              .unstack(fill_value=0))
STOP_TABLE.to_csv(os.path.join(M1_OUTPUT_DIR, "stop_reasons.csv"))
STOP_TABLE


In [ ]:
# --- Figure 7 : what the test agent actually did ------------------------------
if len(TA_TABLE):
    fig, axes = plt.subplots(1, 2, figsize=(4 + 3.2 * len(TA_TABLE), 4.3))
    x = np.arange(len(TA_TABLE))
    ticks = ["%s\n(%d with tests)" % (r["benchmark"], r["questions_with_generated_tests"])
             for r in TA_TABLE.to_dict("records")]

    ax = axes[0]
    g = ax.bar(x - 0.19, TA_TABLE["caught_real_bug"], width=0.36, color=C_GOOD, linewidth=0)
    b = ax.bar(x + 0.19, TA_TABLE["false_alarm_on_correct_draft"], width=0.36,
               color=C_CRITICAL, linewidth=0)
    top = max(1, int(TA_TABLE[["caught_real_bug", "false_alarm_on_correct_draft"]].values.max()))
    ax.set_ylim(0, top * 1.3)
    label_bars(ax, g, "{:.0f}"); label_bars(ax, b, "{:.0f}")
    ax.set_xticks(x); ax.set_xticklabels(ticks, fontsize=8.5)
    ax.set_ylabel("questions"); ax.grid(axis="x", visible=False)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_title("Generated tests flagged the first draft", color=INK, loc="left")
    ax.legend(handles=[Patch(facecolor=C_GOOD, label="caught a bug the public tests missed"),
                       Patch(facecolor=C_CRITICAL, label="false alarm on a correct draft")],
              loc="upper left", fontsize=8.5)

    ax = axes[1]
    v = TA_TABLE["valid_tests_%"].to_numpy(float)
    bars = ax.bar(x, np.nan_to_num(v), width=0.5, color=SYS_COLOR["repair_tests"],
                  hatch=SYS_HATCH["repair_tests"], edgecolor="white", linewidth=0.6)
    ax.set_ylim(0, 110)
    for xi, val, n in zip(x, v, TA_TABLE["tests_checked_vs_reference"]):
        ax.text(xi, (0 if val != val else val) + 2,
                "no reference" if val != val else "%.0f%%\nof %d" % (val, n),
                ha="center", va="bottom", fontsize=8, color=INK_2)
    ax.set_xticks(x); ax.set_xticklabels(ticks, fontsize=8.5)
    ax.set_ylabel("% of generated tests"); ax.grid(axis="x", visible=False)
    ax.set_title("Generated tests that are correct", color=INK, loc="left")

    fig.suptitle("Test agent diagnostics", x=0.05, ha="left", fontsize=12.5, color=INK, y=1.08)
    fig.text(0.05, 1.005, "A generated test is correct when the benchmark's reference solution "
             "passes it. LiveCodeBench has no reference, and its stdin problems get no tests.",
             ha="left", fontsize=9, color=INK_2)
    fig.subplots_adjust(wspace=0.25)
    save(fig, "fig_07_test_agent.png")


## 21 · Cost of each agent inside each system

In [ ]:
# --- cost of each agent, per question, inside each system --------------------
CALLS = pd.concat([pd.DataFrame(R["calls"]).assign(benchmark=t, bench_label=R["label"])
                   for t, R in M1_RESULTS.items() if len(R["calls"])], ignore_index=True)
CALLS["total_tokens"] = CALLS["prompt_tokens"] + CALLS["completion_tokens"]
CALLS["cached"] = CALLS["cached"].astype(bool)

agent_rows, check_rows = [], []
for tag in BENCH_TAGS:
    nq = M1_RESULTS[tag]["df"].task_id.nunique()
    for s in SYSTEMS:
        cs = CALLS[(CALLS.benchmark == tag) & (CALLS.system == s)]
        tot_gpu, tot_j = cs.gpu_seconds.sum(), cs.joules.sum()
        for role in ROLE_KEYS:
            a = cs[cs.role == role]
            fresh = a[~a.cached]           # per-call averages use real measurements only
            n = len(a)
            agent_rows.append({
                "benchmark": M1_RESULTS[tag]["label"], "tag": tag, "system": s, "role": role,
                "agent": ROLE_LABEL[role], "calls": n, "calls_per_question": n / nq,
                "tokens_per_question": a.total_tokens.sum() / nq,
                "gpu_s_per_question": a.gpu_seconds.sum() / nq,
                "joules_per_question": a.joules.sum() / nq,
                "prompt_tokens_per_call": fresh.prompt_tokens.mean() if len(fresh) else np.nan,
                "completion_tokens_per_call": fresh.completion_tokens.mean() if len(fresh) else np.nan,
                "gpu_s_per_call": fresh.gpu_seconds.mean() if len(fresh) else np.nan,
                "joules_per_call": fresh.joules.mean() if len(fresh) else np.nan,
                "share_gpu_s_%": 100.0 * a.gpu_seconds.sum() / tot_gpu if tot_gpu else np.nan,
                "share_energy_%": 100.0 * a.joules.sum() / tot_j if tot_j else np.nan,
            })
        d = ALL[(ALL.benchmark == tag) & (ALL.system == s)]
        check_rows.append((M1_RESULTS[tag]["label"], s, cs.gpu_seconds.sum(), d.gpu_seconds.sum(),
                           len(cs), int(d.calls_total.sum())))

AGENT_TABLE = pd.DataFrame(agent_rows)
AGENT_TABLE.to_csv(os.path.join(M1_OUTPUT_DIR, "per_agent_costs.csv"), index=False)
for lab, s, g1, g2, c1, c2 in check_rows:
    ok = abs(g1 - g2) < 1e-6 and c1 == c2
    print("%-14s %-20s per-agent sums %s the per-question totals (%.1f GPU-s, %d calls)"
          % (lab, SYSTEM_LABEL[s], "match" if ok else "DO NOT MATCH", g2, c2))

AGENT_TABLE[AGENT_TABLE.calls > 0][
    ["benchmark", "system", "agent", "calls_per_question", "tokens_per_question",
     "gpu_s_per_question", "joules_per_question", "prompt_tokens_per_call",
     "completion_tokens_per_call", "gpu_s_per_call", "share_gpu_s_%", "share_energy_%"]].round(2)


In [ ]:
# --- Figure 8 : where each system's cost goes, per question -------------------
panels = [("calls_per_question", "Model calls / question", "{:.2f}"),
          ("tokens_per_question", "Tokens / question", "{:.0f}"),
          ("gpu_s_per_question", "GPU-seconds / question", "{:.1f}"),
          ("joules_per_question", "Joules / question", "{:.0f}")]
used_roles = [r for r in ROLE_KEYS if AGENT_TABLE[AGENT_TABLE.role == r].calls.sum() > 0]
SHORT_SYS = {"baseline": "base", "repair": "rep", "repair_tests": "rep+T"}

fig, axes = plt.subplots(len(panels), 1, figsize=(2.2 + 2.4 * len(BENCH_TAGS) * N_SYS / 3, 3.0 * len(panels)))
pos, ticks, centers = [], [], []
for bi, tag in enumerate(BENCH_TAGS):
    base = bi * (N_SYS + 1)
    for k, s in enumerate(SYSTEMS):
        pos.append((tag, s, base + k)); ticks.append(SHORT_SYS[s])
    centers.append(base + (N_SYS - 1) / 2)
for ax, (col, title, fmt) in zip(axes, panels):
    bottoms = {p: 0.0 for p in pos}
    for role in used_roles:
        vals = []
        for tag, s, xpos in pos:
            r = AGENT_TABLE[(AGENT_TABLE.tag == tag) & (AGENT_TABLE.system == s) &
                            (AGENT_TABLE.role == role)]
            v = float(r[col].iloc[0]) if len(r) else 0.0
            vals.append(0.0 if v != v else v)
        ax.bar([p[2] for p in pos], vals, bottom=[bottoms[p] for p in pos], width=0.8,
               color=ROLE_COLOR[role], edgecolor="white", linewidth=1.0)
        for p, v in zip(pos, vals):
            bottoms[p] += v
    top = max(bottoms.values()) or 1.0
    ax.set_ylim(0, top * 1.2)
    for p in pos:
        ax.text(p[2], bottoms[p] + 0.02 * top, fmt.format(bottoms[p]), ha="center",
                va="bottom", fontsize=7.5, color=INK_2)
    ax.set_xticks([p[2] for p in pos]); ax.set_xticklabels(ticks, fontsize=8)
    for c, tag in zip(centers, BENCH_TAGS):
        ax.text(c, -0.2, M1_RESULTS[tag]["label"], transform=ax.get_xaxis_transform(),
                ha="center", va="top", fontsize=9, color=INK)
    ax.set_title(title, color=INK, loc="left"); ax.grid(axis="x", visible=False)
fig.suptitle("Where each system's cost goes", x=0.05, ha="left", fontsize=12.5, color=INK, y=0.998)
fig.text(0.05, 0.968, "base = baseline, rep = repair, rep+T = repair + test agent. The blue "
         "segment is the shared first draft.", ha="left", fontsize=9, color=INK_2)
fig.legend(handles=[Patch(facecolor=ROLE_COLOR[r], label=ROLE_LABEL[r]) for r in used_roles],
           loc="upper left", bbox_to_anchor=(0.045, 0.962), ncol=len(used_roles), fontsize=9)
fig.subplots_adjust(top=0.91, hspace=0.62)
save(fig, "fig_08_cost_by_agent.png")


In [ ]:
# --- Figure 9 : what ONE call of each agent costs (fresh calls only) ----------
PER_CALL = (CALLS[~CALLS.cached].groupby(["benchmark", "role"])
            [["prompt_tokens", "completion_tokens", "gpu_seconds", "joules"]].mean().reset_index())
panels = [("prompt_tokens", "Prompt tokens / call", "{:.0f}"),
          ("completion_tokens", "Completion tokens / call", "{:.0f}"),
          ("gpu_seconds", "GPU-seconds / call", "{:.2f}"),
          ("joules", "Joules / call", "{:.0f}")]
fig, axes = plt.subplots(2, 2, figsize=(4 + 3.2 * len(BENCH_TAGS), 8.4))
nr = len(used_roles)
w = 0.8 / nr
x = np.arange(len(BENCH_TAGS))
for ax, (col, title, fmt) in zip(axes.ravel(), panels):
    for k, role in enumerate(used_roles):
        vals = []
        for tag in BENCH_TAGS:
            r = PER_CALL[(PER_CALL.benchmark == tag) & (PER_CALL.role == role)]
            vals.append(float(r[col].iloc[0]) if len(r) else np.nan)
        bars = ax.bar(x + (k - (nr - 1) / 2) * w, np.nan_to_num(vals), width=w * 0.92,
                      color=ROLE_COLOR[role], edgecolor="white", linewidth=0.6)
        bars._m1_vals = vals
    ax.set_ylim(0, ax.get_ylim()[1] * 1.15)
    for c in ax.containers:
        label_bars(ax, c, fmt, c._m1_vals)
    ax.set_xticks(x); ax.set_xticklabels(BENCH_LABELS)
    ax.set_title(title, color=INK, loc="left"); ax.grid(axis="x", visible=False)
fig.legend(handles=[Patch(facecolor=ROLE_COLOR[r], label=ROLE_LABEL[r]) for r in used_roles],
           loc="upper left", bbox_to_anchor=(0.045, 0.95), ncol=nr, fontsize=9)
fig.suptitle("What one call of each agent costs", x=0.05, ha="left", fontsize=12.5, color=INK, y=1.0)
fig.text(0.05, 0.957, "Averages over calls actually sent to the GPU; cached replays of the shared "
         "first draft are excluded.", ha="left", fontsize=9, color=INK_2)
fig.subplots_adjust(top=0.87, hspace=0.3, wspace=0.18)
save(fig, "fig_09_cost_per_call.png")


## 22 · What drives cost: calls, tokens, GPU time and energy

The first M1 run fitted one straight line through every question of every system and benchmark. That
overstated what the data shows, for five reasons, and this section replaces it:

1. **Leverage.** Most questions sit at exactly 1 call; a handful at 7 calls set the slope.
2. **Pooled benchmarks.** A HumanEval+ call costs ~3x an MBPP+ call, so one pooled "GPU-s per call" is true for neither.
3. **Baseline rows.** The baseline always makes 1 call, so including it mostly measures "baseline vs repair".
4. **Mechanical pairs.** GPU-seconds and wall time come from the same timer, and joules are measured over that
   window, so r ≈ 1 between them is by construction. They are reported below as **power**, not as correlations.
5. **Prompt and completion tokens were summed.** Reading a prompt (prefill) and writing tokens (decode) cost very
   different amounts per token.

What replaces it:

- **(A) Per-call cost model** on calls actually sent to the GPU (cached replays excluded), fitted per benchmark:
  `GPU-s = a + b_prompt · prompt_k + b_completion · completion_k` (same for joules). The coefficients are the
  price of 1,000 prompt tokens and 1,000 completion tokens.
- **(B) Power** by role: joules per wall-second while generating, minus idle.
- **(C) Per question**, repair systems only, per benchmark: Spearman ρ with a 95% bootstrap interval between each
  count (calls, calls + test runs, prompt tokens, completion tokens) and each cost (GPU-s, joules), plus how much of
  the GPU-second variance calls alone vs tokens explain.


In [ ]:
rng_boot = np.random.default_rng(SEED)
FRESH = CALLS[~CALLS.cached].copy()
FRESH["prompt_k"] = FRESH.prompt_tokens / 1000.0
FRESH["completion_k"] = FRESH.completion_tokens / 1000.0


def ols(X, y):
    """Least squares with intercept -> (coefs incl. intercept, R^2)."""
    X = np.column_stack([np.ones(len(X))] + [np.asarray(c, float) for c in X.T])
    y = np.asarray(y, float)
    ok = np.isfinite(X).all(axis=1) & np.isfinite(y)
    X, y = X[ok], y[ok]
    if len(y) < X.shape[1] + 2 or np.ptp(y) == 0:
        return np.full(X.shape[1], np.nan), np.nan
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta
    r2 = 1 - (resid ** 2).sum() / ((y - y.mean()) ** 2).sum()
    return beta, float(r2)


# ---- (A) per-call cost model -------------------------------------------------
cm_rows = []
for tag in BENCH_TAGS + ["__pooled__"]:
    d = FRESH if tag == "__pooled__" else FRESH[FRESH.benchmark == tag]
    for target in ["gpu_seconds", "joules"]:
        beta2, r2_two = ols(d[["prompt_k", "completion_k"]].to_numpy(), d[target])
        _, r2_total = ols((d.prompt_k + d.completion_k).to_numpy()[:, None], d[target])
        cm_rows.append({
            "benchmark": "all (pooled)" if tag == "__pooled__" else M1_RESULTS[tag]["label"],
            "target": target, "calls": len(d),
            "intercept_per_call": beta2[0],
            "per_1k_prompt_tokens": beta2[1], "per_1k_completion_tokens": beta2[2],
            "completion_over_prompt": beta2[2] / beta2[1] if beta2[1] and beta2[1] > 0 else np.nan,
            "r2_prompt+completion": r2_two, "r2_total_tokens": r2_total})
COST_MODEL = pd.DataFrame(cm_rows)
COST_MODEL.to_csv(os.path.join(M1_OUTPUT_DIR, "cost_model_per_call.csv"), index=False)

# ---- (B) power by role -------------------------------------------------------
idle = M1_CONFIG["IDLE_WATTS"]
FRESH["power_w"] = FRESH.joules / FRESH.wall_seconds.replace(0, np.nan)
POWER = (FRESH.groupby(["bench_label", "role"])
         .agg(calls=("power_w", "size"), mean_power_w=("power_w", "mean"),
              p10=("power_w", lambda s: s.quantile(0.1)), p90=("power_w", lambda s: s.quantile(0.9)))
         .reset_index())
POWER["above_idle_w"] = POWER.mean_power_w - idle
POWER["per_card_w"] = POWER.mean_power_w / max(M1_CONFIG["N_GPUS"], 1)
POWER.to_csv(os.path.join(M1_OUTPUT_DIR, "power_by_role.csv"), index=False)

# ---- (C) per question, repair systems only -----------------------------------
try:
    from scipy.stats import rankdata as _rank
except Exception:
    _rank = lambda a: pd.Series(a).rank().to_numpy()

def spearman(x, y):
    x = _rank(x); y = _rank(y)
    if np.ptp(x) == 0 or np.ptp(y) == 0:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])

def spearman_ci(x, y, B=500):
    x, y = np.asarray(x, float), np.asarray(y, float)
    ok = np.isfinite(x) & np.isfinite(y); x, y = x[ok], y[ok]
    if len(x) < 8:
        return np.nan, np.nan, np.nan
    est = spearman(x, y)
    idx = rng_boot.integers(0, len(x), size=(B, len(x)))
    boots = [spearman(x[i], y[i]) for i in idx]
    boots = [b for b in boots if b == b]
    lo, hi = (np.percentile(boots, [2.5, 97.5]) if boots else (np.nan, np.nan))
    return est, float(lo), float(hi)

COUNTS = [("calls_total", "model calls"), ("calls_plus_exec", "calls + test runs"),
          ("prompt_tokens", "prompt tokens"), ("completion_tokens", "completion tokens"),
          ("total_tokens", "total tokens")]
COSTS = [("gpu_seconds", "GPU-seconds"), ("joules", "joules")]
REP = ALL[ALL.system != "baseline"]

rel_rows = []
for tag in BENCH_TAGS:
    for scope, d in [("all questions", REP[REP.benchmark == tag]),
                     ("entered repair", REP[(REP.benchmark == tag) & (REP.iterations > 0)])]:
        for cx, cxl in COUNTS:
            for cy, cyl in COSTS:
                rho, lo, hi = spearman_ci(d[cx], d[cy])
                _, r2 = ols(d[[cx]].to_numpy(float), d[cy])
                rel_rows.append({"benchmark": M1_RESULTS[tag]["label"], "tag": tag, "scope": scope,
                                 "n": len(d), "x": cx, "x_label": cxl, "y": cy, "y_label": cyl,
                                 "spearman": rho, "ci_lo": lo, "ci_hi": hi, "linear_r2": r2})
RELATIONS = pd.DataFrame(rel_rows)
RELATIONS.to_csv(os.path.join(M1_OUTPUT_DIR, "metric_relationships.csv"), index=False)

# ---- plain-language read-out -------------------------------------------------
print("(A) Per-call cost model, fresh calls only")
for r in COST_MODEL[COST_MODEL.target == "gpu_seconds"].to_dict("records"):
    print("  %-14s %5d calls: %.2f GPU-s per 1k prompt tokens, %.2f per 1k completion tokens "
          "(decode %.0fx prefill); R^2 %.2f with the split vs %.2f with total tokens"
          % (r["benchmark"], r["calls"], r["per_1k_prompt_tokens"], r["per_1k_completion_tokens"],
             r["completion_over_prompt"], r["r2_prompt+completion"], r["r2_total_tokens"]))
print("\n(B) Power while generating (all %d cards; idle %.0f W)" % (M1_CONFIG["N_GPUS"], idle))
for r in POWER.to_dict("records"):
    print("  %-14s %-26s %.0f W  (%.0f W above idle, 10-90%%: %.0f-%.0f W)"
          % (r["bench_label"], ROLE_LABEL.get(r["role"], r["role"]), r["mean_power_w"],
             r["above_idle_w"], r["p10"], r["p90"]))
print("\n(C) Per question, repair systems: what explains GPU-seconds?")
for tag in BENCH_TAGS:
    sub = RELATIONS[(RELATIONS.tag == tag) & (RELATIONS.scope == "all questions") &
                    (RELATIONS.y == "gpu_seconds")].set_index("x")
    print("  %-14s calls rho %.2f [%.2f, %.2f], R^2 %.2f | total tokens rho %.2f [%.2f, %.2f], R^2 %.2f"
          % (M1_RESULTS[tag]["label"],
             sub.loc["calls_total", "spearman"], sub.loc["calls_total", "ci_lo"],
             sub.loc["calls_total", "ci_hi"], sub.loc["calls_total", "linear_r2"],
             sub.loc["total_tokens", "spearman"], sub.loc["total_tokens", "ci_lo"],
             sub.loc["total_tokens", "ci_hi"], sub.loc["total_tokens", "linear_r2"]))
COST_MODEL.round(3)


In [ ]:
# --- Figure 10 : the per-call cost model --------------------------------------
fig, axes = plt.subplots(2, len(BENCH_TAGS), figsize=(4.8 * len(BENCH_TAGS), 8.2), squeeze=False)
for j, tag in enumerate(BENCH_TAGS):
    d = FRESH[FRESH.benchmark == tag]
    cm = COST_MODEL[(COST_MODEL.benchmark == M1_RESULTS[tag]["label"]) &
                    (COST_MODEL.target == "gpu_seconds")].iloc[0]
    for i, (xcol, xl) in enumerate([("completion_tokens", "completion tokens per call"),
                                    ("prompt_tokens", "prompt tokens per call")]):
        ax = axes[i][j]
        for role in used_roles:
            s = d[d.role == role]
            ax.scatter(s[xcol], s.gpu_seconds, s=16, color=ROLE_COLOR[role], alpha=0.75,
                       edgecolor="white", linewidth=0.4, zorder=3)
        ax.set_xlabel(xl)
        ax.set_xlim(left=0); ax.set_ylim(bottom=0)
        if j == 0:
            ax.set_ylabel("GPU-seconds per call")
        if i == 0:
            ax.set_title("%s (%d calls)" % (M1_RESULTS[tag]["label"], len(d)), color=INK, loc="left")
            ax.text(0.03, 0.97, "per 1k completion tokens: %.2f GPU-s\nper 1k prompt tokens: %.2f GPU-s\n"
                    "R$^2$ = %.2f (split)  vs  %.2f (total)"
                    % (cm["per_1k_completion_tokens"], cm["per_1k_prompt_tokens"],
                       cm["r2_prompt+completion"], cm["r2_total_tokens"]),
                    transform=ax.transAxes, ha="left", va="top", fontsize=8.5, color=INK,
                    bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor=GRID))
fig.suptitle("What one call costs: prompt tokens vs completion tokens",
             x=0.04, ha="left", fontsize=12.5, color=INK, y=1.0)
fig.text(0.04, 0.957, "One point per call actually sent to the GPU. Coefficients come from one "
         "fit with both token counts, per benchmark.", ha="left", fontsize=9, color=INK_2)
fig.legend(handles=[Patch(facecolor=ROLE_COLOR[r], label=ROLE_LABEL[r]) for r in used_roles],
           loc="upper left", bbox_to_anchor=(0.035, 0.95), ncol=len(used_roles), fontsize=9)
fig.subplots_adjust(top=0.87, hspace=0.3, wspace=0.22)
save(fig, "fig_10_cost_model_per_call.png")


In [ ]:
# --- Figure 11 : per question, repair systems - calls vs tokens as cost proxies
arms = [s for s in SYSTEMS if s != "baseline"]
rng_j = np.random.default_rng(SEED)
fig, axes = plt.subplots(len(BENCH_TAGS), 2, figsize=(11, 3.7 * len(BENCH_TAGS)), squeeze=False)
for i, tag in enumerate(BENCH_TAGS):
    d = REP[REP.benchmark == tag]
    sub = RELATIONS[(RELATIONS.tag == tag) & (RELATIONS.scope == "all questions") &
                    (RELATIONS.y == "gpu_seconds")].set_index("x")
    for j, (xcol, xl) in enumerate([("calls_total", "model calls"), ("total_tokens", "tokens")]):
        ax = axes[i][j]
        for s in arms:
            ds = d[d.system == s]
            xv = ds[xcol].to_numpy(float)
            if xcol == "calls_total":
                xv = xv + rng_j.uniform(-0.15, 0.15, len(xv))
            ax.scatter(xv, ds.gpu_seconds, s=20, color=SYS_COLOR[s], alpha=0.7,
                       marker="o" if s != "repair_tests" else "D",
                       edgecolor="white", linewidth=0.5, zorder=3)
        r = sub.loc[xcol]
        ax.text(0.03, 0.97, "Spearman rho %.2f [%.2f, %.2f]\nlinear R$^2$ %.2f"
                % (r["spearman"], r["ci_lo"], r["ci_hi"], r["linear_r2"]),
                transform=ax.transAxes, ha="left", va="top", fontsize=8.5, color=INK,
                bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor=GRID))
        ax.set_xlabel(xl); ax.set_xlim(left=0); ax.set_ylim(bottom=0)
        if j == 0:
            ax.set_ylabel("%s\nGPU-seconds per question" % M1_RESULTS[tag]["label"])
handles = [plt.Line2D([], [], marker="o" if s != "repair_tests" else "D", linestyle="none",
                      markerfacecolor=SYS_COLOR[s], markeredgecolor="white", markersize=8,
                      label=SYSTEM_LABEL[s]) for s in arms]
fig.legend(handles=handles, loc="upper right", bbox_to_anchor=(0.995, 1.02), ncol=len(arms))
fig.suptitle("Per question: call count vs tokens as a proxy for GPU time",
             x=0.05, ha="left", fontsize=12.5, color=INK, y=1.02)
fig.text(0.05, 0.99 - 0.004 * len(BENCH_TAGS), "Repair systems only, one row per benchmark, "
         "95% bootstrap intervals. Call counts are jittered.", ha="left", fontsize=9, color=INK_2)
fig.subplots_adjust(top=0.93, hspace=0.35, wspace=0.2)
save(fig, "fig_11_calls_vs_tokens.png")


In [ ]:
# --- Figure 12 : Spearman correlations per benchmark (repair systems) ---------
from matplotlib.colors import LinearSegmentedColormap

VARS = [("calls_total", "calls"), ("calls_plus_exec", "calls + runs"),
        ("prompt_tokens", "prompt tok"), ("completion_tokens", "completion tok"),
        ("gpu_seconds", "GPU-s"), ("joules", "joules")]
CORRS = {}
for tag in BENCH_TAGS:
    d = REP[REP.benchmark == tag][[v for v, _ in VARS]]
    CORRS[tag] = d.corr(method="spearman")
    CORRS[tag].to_csv(os.path.join(M1_OUTPUT_DIR, "spearman_%s.csv" % tag))

allv = np.concatenate([c.values[np.tril_indices(len(VARS), -1)] for c in CORRS.values()])
allv = allv[np.isfinite(allv)]
if len(allv) and allv.min() >= 0:           # one-sided data -> sequential blue ramp
    cmap = LinearSegmentedColormap.from_list("seq", ["#cde2fb", "#86b6ef", "#2a78d6", "#104281"])
    vmin, vmax, cbl = 0, 1, "Spearman rho"
else:                                       # signs present -> diverging, grey midpoint
    cmap = LinearSegmentedColormap.from_list("div", ["#2a78d6", "#f0efec", "#e34948"])
    vmin, vmax, cbl = -1, 1, "Spearman rho"

k = len(VARS)
labs = [l for _, l in VARS]
fig, axes = plt.subplots(1, len(BENCH_TAGS), figsize=(4.9 * len(BENCH_TAGS), 4.9), squeeze=False)
for ax, tag in zip(axes[0], BENCH_TAGS):
    vals = CORRS[tag].values
    mask = np.triu(np.ones((k, k), dtype=bool))
    im = ax.imshow(np.ma.array(vals, mask=mask)[1:, :-1], cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(range(k - 1)); ax.set_xticklabels(labs[:-1], rotation=35, ha="right", fontsize=8.5)
    ax.set_yticks(range(k - 1)); ax.set_yticklabels(labs[1:], fontsize=8.5)
    ax.grid(False)
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.set_xticks(np.arange(-0.5, k - 1, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, k - 1, 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=2); ax.tick_params(which="minor", length=0)
    for i in range(1, k):
        for j in range(i):
            v = vals[i, j]
            if v == v:
                ax.text(j, i - 1, "%.2f" % v, ha="center", va="center", fontsize=8.5,
                        color="white" if v > 0.6 else INK)
    ax.set_title("%s (n = %d)" % (M1_RESULTS[tag]["label"], int((REP.benchmark == tag).sum())),
                 color=INK, loc="left")
cb = fig.colorbar(im, ax=list(axes[0]), fraction=0.02, pad=0.02)
cb.set_label(cbl, color=INK_2); cb.outline.set_visible(False)
fig.suptitle("Rank correlations between cost measures, repair systems", x=0.04, ha="left",
             fontsize=12.5, color=INK, y=1.03)
fig.text(0.04, 0.975, "GPU-s and joules come from the same measurement window, so their "
         "correlation is expected; compare the calls and token columns instead.",
         ha="left", fontsize=9, color=INK_2)
save(fig, "fig_12_spearman_matrix.png")


## 23 · Summary report

In [ ]:
def _f(v, fmt):
    return "-" if v is None or (isinstance(v, float) and v != v) else fmt % v

L = ["# Milestone 1 (v2) results - %s" % MODEL_NAME, "",
     "%s - %d GPU(s) - idle %s - repair feedback: %s tests - graded on hidden tests" % (
         time.strftime("%Y-%m-%d %H:%M"), M1_CONFIG["N_GPUS"],
         _f(M1_CONFIG["IDLE_WATTS"], "%.1f W"), FEEDBACK_TESTS), ""]
for t, R in M1_RESULTS.items():
    L.append("- %s: %d questions%s" % (R["label"], R["df"].task_id.nunique(),
                                       "" if R["complete"] else " (PARTIAL)"))

L += ["", "## Correctness and cost per question", "",
      "| benchmark | system | n | pass@1 (95% CI) | calls | calls + runs | tokens | GPU-s | J | GPU-s per solved | entered repair | false accept |",
      "|---|---|---|---|---|---|---|---|---|---|---|---|"]
for r in M1_METRICS.to_dict("records"):
    L.append("| %s | %s | %d | %.1f (%.1f-%.1f) | %.2f | %.2f | %.0f | %.1f | %s | %s | %.0f%% | %s |"
             % (r["benchmark"], SYSTEM_LABEL[r["system"]], r["n_tasks"], r["pass@1"], r["ci_lo"],
                r["ci_hi"], r["calls_per_task"], r["calls_plus_exec_per_task"], r["tokens_per_task"],
                r["gpu_s_per_task"], _f(r["joules_per_task"], "%.0f"),
                _f(r["gpu_s_per_solved"], "%.1f"), r["entered_repair_%"],
                _f(r["false_accept_%"], "%.0f%%")))

L += ["", "## Paired comparisons (exact McNemar, Holm-corrected)", "",
      "| benchmark | comparison | n | delta (pp) | gains | regressions | p | p (Holm) | do-no-harm |",
      "|---|---|---|---|---|---|---|---|---|"]
for r in M1_STATS.to_dict("records"):
    L.append("| %s | %s | %d | %+.1f | %d | %d | %.4f | %.4f | %s |"
             % (r["benchmark"], r["comparison"], r["n"], r["delta_pp"], r["gains"],
                r["regressions"], r["p_exact"], r["p_holm"],
                "pass" if r["do_no_harm_pass"] else "FAIL"))

L += ["", "## Test agent", "",
      "| benchmark | questions with tests | tests/question | correct tests | caught real bug | false alarm | extra GPU-s vs repair |",
      "|---|---|---|---|---|---|---|"]
for r in TA_TABLE.to_dict("records"):
    L.append("| %s | %d / %d | %s | %s | %d | %d | %+.2f |"
             % (r["benchmark"], r["questions_with_generated_tests"], r["questions"],
                _f(r["tests_per_question"], "%.1f"), _f(r["valid_tests_%"], "%.0f%%"),
                r["caught_real_bug"], r["false_alarm_on_correct_draft"], r["extra_gpu_s_vs_repair"]))

L += ["", "## Per-call cost model (fresh calls)", "",
      "| benchmark | target | calls | per 1k prompt tok | per 1k completion tok | R^2 split | R^2 total tokens |",
      "|---|---|---|---|---|---|---|"]
for r in COST_MODEL.to_dict("records"):
    L.append("| %s | %s | %d | %.3f | %.3f | %.2f | %.2f |"
             % (r["benchmark"], r["target"], r["calls"], r["per_1k_prompt_tokens"],
                r["per_1k_completion_tokens"], r["r2_prompt+completion"], r["r2_total_tokens"]))

L += ["", "## Per question, repair systems: Spearman rho with GPU-seconds", "",
      "| benchmark | scope | n | calls | calls + runs | total tokens |", "|---|---|---|---|---|---|"]
for (b, sc), g in RELATIONS[RELATIONS.y == "gpu_seconds"].groupby(["benchmark", "scope"], sort=False):
    gi = g.set_index("x")
    cell = lambda x: "%.2f [%.2f, %.2f]" % (gi.loc[x, "spearman"], gi.loc[x, "ci_lo"], gi.loc[x, "ci_hi"])
    L.append("| %s | %s | %d | %s | %s | %s |" % (b, sc, int(gi["n"].iloc[0]), cell("calls_total"),
                                                   cell("calls_plus_exec"), cell("total_tokens")))

SUMMARY_MD = "\n".join(L)
with open(os.path.join(M1_OUTPUT_DIR, "summary.md"), "w") as f:
    f.write(SUMMARY_MD)
print(SUMMARY_MD)


## 24 · Download all results

This last cell zips everything in `outputs_milestone1/` — CSVs, per-call logs, figures and the
summary — into **`milestone1_results.zip`**.

- **Kaggle:** click the link printed below, or open the **Output** panel on the right. Files in
  `/kaggle/working` are deleted when the session ends unless you download them or press
  *Save Version*.
- **Colab:** the download starts automatically.

In [ ]:
import zipfile

ZIP_PATH = os.path.join(BASE_DIR, "milestone1_results.zip")
n_files = 0
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(M1_OUTPUT_DIR):
        for fn in sorted(files):
            full = os.path.join(root, fn)
            z.write(full, os.path.relpath(full, BASE_DIR))
            n_files += 1

print("Packed %d files -> %s  (%.1f MB)"
      % (n_files, os.path.abspath(ZIP_PATH), os.path.getsize(ZIP_PATH) / 1e6))
with zipfile.ZipFile(ZIP_PATH) as z:
    for name in z.namelist():
        print("   ", name)

if globals().get("IS_COLAB"):
    from google.colab import files
    files.download(ZIP_PATH)
else:
    try:
        from IPython.display import FileLink, display
        print("\nClick to download:")
        display(FileLink(os.path.relpath(ZIP_PATH)))
    except Exception:
        print("\nDownload it from:", os.path.abspath(ZIP_PATH))
    if ON_KAGGLE:
        print("On Kaggle it is also listed in the Output panel (right sidebar) -> milestone1_results.zip")

---
### Measurement notes
- **Grading is always on the hidden suite.** With `FEEDBACK_TESTS = "public"` no agent ever sees a hidden test.
- **GPU-seconds are occupancy**, not utilisation: server compute time × GPUs holding the model. Model load is
  excluded, and a warm-up call runs before each benchmark so no load lands in a measured call.
- **Energy is GPU-card energy** from NVML's hardware counter, over each call's wall window. CPU, RAM and PSU losses
  are excluded, so it is a lower bound on machine energy, measured the same way for every system.
- **Sandbox execution is CPU work** and is not in GPU-seconds or joules. It enters the cost only through the
  "calls + test runs" unit.
- **Shared first draft.** All systems use the same draft (prompt cache + fixed seed) and each is charged its
  measured cost. Supervisor and repair prompts that are identical across `repair` and `repair_tests` are also
  served from the cache, so the two systems differ only where the generated tests change what the loop sees.
- **LiveCodeBench contamination.** Check the printed contest-date range against the model's training cutoff before
  reading LiveCodeBench numbers as uncontaminated.
